# Brain Tumor Classification — ML-Only (No Deep Learning)

## Architecture: One Model Per Data Modality → Ensemble

```
Radiomics (ax_t1)     → XGBoost / LightGBM / CatBoost / RF / ET / GBM / SVM / LR / KNN → prob(5,)
Radiomics (ax_t1c)    → XGBoost / LightGBM / CatBoost / RF / ET / GBM / SVM / LR / KNN → prob(5,)
Radiomics (ax_t2)     → XGBoost / LightGBM / CatBoost / RF / ET / GBM / SVM / LR / KNN → prob(5,)
Radiomics (ax_t2f)    → XGBoost / LightGBM / CatBoost / RF / ET / GBM / SVM / LR / KNN → prob(5,)
Clinical              → XGBoost / LightGBM / CatBoost / RF / SVM / LR / KNN / LDA        → prob(5,)
Image (PCA 128d)      → XGBoost / LightGBM / CatBoost / RF / ET / GBM / SVM / LR / MLP  → prob(5,)
Report (TF-IDF+SVD)   → LR / SVM / MLP / KNN / XGB / LGB                                  → prob(5,)
Location (636 class)  → XGBoost / LightGBM / CatBoost / RF / ET                             → prob(5,)
    ↓
All prob(5,) vectors → weighted average → final prediction
```

**Small data strategy**: Strong regularization, 5-fold StratifiedKFold CV.  

*AI-assisted tools were used for code debugging and figure visualization. All experimental design, model architecture decisions, and analysis were conducted by the authors.*

# Warning !!! Be careful, this is an experiment use only, not the final model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Brain Tumor Classify"

Mounted at /content/drive
/content/drive/MyDrive/Brain Tumor Classify


In [ ]:

# Mount Google Drive (for Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/Brain Tumor Classify"

import numpy as np
import pandas as pd
import os, warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, os.getcwd())
from data_loader import get_all_data

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

import xgboost as xgb

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

print(f"LightGBM: {HAS_LGB}, CatBoost: {HAS_CAT}, SMOTE: {HAS_SMOTE}")


LightGBM: True, CatBoost: False, SMOTE: True


In [ ]:

# ============================================================
# ENHANCED FEATURES MODULE (inlined from enhanced_features.py)
# ============================================================

"""
Enhanced Radiomics and Image Features - Simplified
Only XGBoost and LightGBM (best performing models)
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

def extract_enhanced_radiomics(data_dict):
    """Enhanced radiomics: 104d from raw 20d."""
    MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']
    RAD_FEAT_NAMES = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
                      'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']

    eps = 1e-8
    result = {}

    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        case_ids = merged['case_id'].values
        N = len(case_ids)

        # Raw radiomics (20d)
        raw_list = []
        for mod in MODALITIES:
            cols = [f'{mod}_{feat}' for feat in RAD_FEAT_NAMES]
            vals = merged[cols].fillna(0).values.astype(np.float32)
            raw_list.append(vals)
        raw = np.hstack(raw_list)

        mod_data = {
            'ax_t1':  raw[:, 0:5], 'ax_t1c': raw[:, 5:10],
            'ax_t2':  raw[:, 10:15], 'ax_t2f': raw[:, 15:20],
        }

        # Cross-modal ratios (25d)
        ratios = []
        for i in range(5):
            enh = (mod_data['ax_t1c'][:, i] + eps) / (mod_data['ax_t1'][:, i] + eps)
            edm = (mod_data['ax_t2f'][:, i] + eps) / (mod_data['ax_t2'][:, i] + eps)
            tc = (mod_data['ax_t1'][:, i] + eps) / (mod_data['ax_t2'][:, i] + eps)
            den = mod_data['ax_t1c'][:, i] - mod_data['ax_t1'][:, i]
            ded = mod_data['ax_t2f'][:, i] - mod_data['ax_t2'][:, i]
            ratios.extend([enh, edm, tc, den, ded])
        ratios = np.clip(np.column_stack(ratios), 0.01, 100.0)
        ratios_log = np.log(ratios)

        # Shape proxies (20d)
        shape_feats = []
        for mod_name, mod_arr in mod_data.items():
            volume_proxy = mod_arr[:, 0] * (1 + mod_arr[:, 3] / 100)
            surface_vol_proxy = mod_arr[:, 1] / (mod_arr[:, 0] + eps)
            compactness_proxy = (mod_arr[:, 0] ** 2) / ((mod_arr[:, 2] + eps) * (mod_arr[:, 1] + eps))
            elong_proxy = np.abs(mod_data['ax_t1c'][:, 3] - mod_data['ax_t1'][:, 3]) / (mod_data['ax_t1'][:, 3] + eps)
            flat_proxy = mod_arr[:, 0] / (mod_arr[:, 1] + eps)
            shape_feats.extend([volume_proxy, surface_vol_proxy, compactness_proxy, elong_proxy, flat_proxy])
        shape_feats = np.column_stack(shape_feats)

        # Texture complexity (10d)
        texture_feats = []
        for mod1, mod2 in [('ax_t1', 'ax_t1c'), ('ax_t2', 'ax_t2f')]:
            texture_feats.append(np.abs(mod_data[mod1][:, 1] - mod_data[mod2][:, 1]))
            texture_feats.append(np.abs(mod_data[mod1][:, 3] - mod_data[mod2][:, 3]))
        for i in [0, 2]:
            texture_feats.append(raw[:, i] * raw[:, 3])  # Mean/90th × Contrast
        texture_feats = np.column_stack(texture_feats[:10])

        # Missing flags (4d)
        missing = np.hstack([merged.get(f'{mod}_missing', pd.Series(0, index=merged.index)).values.reshape(-1, 1)
                             for mod in MODALITIES])

        X = np.hstack([raw, ratios_log, np.log1p(np.abs(shape_feats)), texture_feats, missing])
        result[split] = {'case_ids': case_ids, 'features': X}

    return result


def extract_image_statistics(data_dict):
    """Extract statistics from ResNet embeddings: 56d per sample."""
    MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']

    result = {}
    for split in ['train', 'val', 'test']:
        img_dict = data_dict[split]['image']
        merged = data_dict[split]['merged']
        case_ids = merged['case_id'].values

        emb_dim = next(iter(img_dict.values()))[MODALITIES[0]].shape[0]
        stats_features = []

        for cid in case_ids:
            cid_feats = img_dict.get(cid, {})
            cid_stats = []

            for mod in MODALITIES:
                emb = cid_feats.get(mod, np.zeros(emb_dim, dtype=np.float32))

                # Core statistics
                mean_val = np.mean(emb)
                std_val = np.std(emb)
                min_val = np.min(emb)
                max_val = np.max(emb)
                p25 = np.percentile(emb, 25)
                p75 = np.percentile(emb, 75)
                p90 = np.percentile(emb, 90)
                p10 = np.percentile(emb, 10)

                # Shape statistics
                skew = np.mean((emb - mean_val) ** 3) / (std_val ** 3 + 1e-8) if std_val > 0 else 0
                kurt = np.mean((emb - mean_val) ** 4) / (std_val ** 4 + 1e-8) - 3 if std_val > 0 else 0

                # Energy and entropy
                energy = np.sum(emb ** 2)
                prob = np.abs(emb) / (np.sum(np.abs(emb)) + 1e-8)
                entropy = -np.sum(prob * np.log(prob + 1e-8))
                l2_norm = np.linalg.norm(emb)

                cid_stats.extend([mean_val, std_val, min_val, max_val, p25, p75, p90, p10, skew, kurt, energy, entropy, l2_norm, len(emb)])

            stats_features.append(cid_stats)

        result[split] = {'case_ids': case_ids, 'features': np.array(stats_features, dtype=np.float32)}

    return result


def build_unified_features(data, clin_data, loc_data, img_data, use_enhanced_rad=True, use_img_stats=True):
    """Build unified feature matrix with all sources."""
    from sklearn.preprocessing import OneHotEncoder

    features_list = []

    # Clinical (27d)
    X_clin_train = np.vstack([clin_data['train']['features'], clin_data['val']['features']])
    X_clin_test = data['test']['merged'][['Sex_enc', 'Age_clean', 'Age_missing'] +
                                         [c for c in data['test']['merged'].columns if c.startswith('Signal Intensity')]].fillna(0).values
    features_list.append(('clinical', X_clin_train, X_clin_test))

    # Location OHE
    loc_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_loc_train = loc_ohe.fit_transform(np.concatenate([loc_data['train']['features'], loc_data['val']['features']]).reshape(-1, 1))
    X_loc_test = loc_ohe.transform(data['test']['merged']['Location_enc'].values.astype(int).reshape(-1, 1))
    features_list.append(('location', X_loc_train, X_loc_test))

    # Enhanced radiomics
    if use_enhanced_rad:
        rad_data = extract_enhanced_radiomics(data)
        features_list.append(('radiomics',
                              np.vstack([rad_data['train']['features'], rad_data['val']['features']]),
                              rad_data['test']['features']))

    # Image ResNet (512d)
    X_img_train = np.vstack([img_data['train']['features'], img_data['val']['features']])
    X_img_test = np.zeros((len(data['test']['merged']), 512))
    for k, cid in enumerate(data['test']['merged']['case_id']):
        cid_int = int(cid)
        if cid_int in data['test']['image']:
            for j_mod, mod in enumerate(['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']):
                feat = data['test']['image'][cid_int].get(mod, np.zeros(128, dtype=np.float32))
                X_img_test[k, j_mod*128:(j_mod+1)*128] = feat
    features_list.append(('image', X_img_train, X_img_test))

    # Image statistics
    if use_img_stats:
        img_stats = extract_image_statistics(data)
        features_list.append(('img_stats',
                              np.vstack([img_stats['train']['features'], img_stats['val']['features']]),
                              img_stats['test']['features']))

    X_train = np.hstack([f[1] for f in features_list])
    X_test = np.hstack([f[2] for f in features_list])

    return {
        'train': {'X': X_train, 'y': np.concatenate([data['train']['label'], data['val']['label']])},
        'test': {'X': X_test, 'case_ids': data['test']['merged']['case_id'].values}
    }, features_list



print("Enhanced features module loaded.")


Enhanced features module loaded.


In [ ]:

data = get_all_data('kaggle-dataset', pca_dim=128)

label_encoder = data['label_encoder']
NUM_CLASSES = len(label_encoder.classes_)
MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']
RAD_FEAT_NAMES = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
                  'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']

y_train = data['train']['label']
y_val   = data['val']['label']
y_tv    = np.concatenate([y_train, y_val])

print(f"\nTrain: {len(y_train)}, Val: {len(y_val)}, Train+Val: {len(y_tv)}")
print(f"Classes: {list(label_encoder.classes_)}")
for i, cls in enumerate(label_encoder.classes_):
    n = (y_tv == i).sum()
    print(f"  {i}: {str(cls)[:40]:40s} n={n:4d} ({100*n/len(y_tv):.1f}%)")


Loading all data...
Using PCA: 128 dimensions per modality (from 2048)
[Clinical] 27 features + 1 location (embedding), age_median=51.0, locations=636
[Reports] train=1983, val=283, test=572
[Radiomics] 20 features + 4 missing flags = 24 total
[Image] Loading from cache with PCA(128)...
[PCA] Fitting PCA on 7932 samples (1983 cases × 4 modalities)
[PCA] 128 components explain 94.16% variance
[Image] 128-dim (PCA from 2048) x 4 modalities, train=1983, val=283, test=572
[Labels] 2266 labeled samples, 5 classes
Label classes: [np.str_('Brain Metastase Tumour'), np.str_('Glioma'), np.str_('Meningioma'), np.str_('Pineal tumour and Choroid plexus tumour'), np.str_('Tumors of the sellar region')]
Train: 1983 | Val: 283 | Test: 572
Image dim: 128 (PCA-reduced from 2048)
All data loaded successfully.

Train: 1983, Val: 283, Train+Val: 2266
Classes: [np.str_('Brain Metastase Tumour'), np.str_('Glioma'), np.str_('Meningioma'), np.str_('Pineal tumour and Choroid plexus tumour'), np.str_('Tumors of

In [ ]:

# ---- 3a. Radiomics per modality ----
def extract_radiomics(data_dict):
    result = {}
    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        split_data = {'case_ids': merged['case_id'].values}
        for mod in MODALITIES:
            cols = [f'{mod}_{feat}' for feat in RAD_FEAT_NAMES]
            vals = merged[cols].fillna(0).values.astype(np.float32)
            missing = merged.get(f'{mod}_missing', pd.Series(0, index=merged.index)).values.reshape(-1, 1).astype(np.float32)
            split_data[mod] = np.hstack([vals, missing])
        result[split] = split_data
    return result

rad_data = extract_radiomics(data)

# ---- 3b. Clinical ----
def extract_clinical(data_dict):
    result = {}
    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        clin_cols = ['Sex_enc', 'Age_clean', 'Age_missing']
        si_cols = [c for c in merged.columns if c.startswith('Signal Intensity')]
        all_cols = clin_cols + si_cols
        vals = merged[all_cols].fillna(0).values.astype(np.float32)
        result[split] = {'case_ids': merged['case_id'].values, 'features': vals}
    return result

clin_data = extract_clinical(data)

# ---- 3c. Location ----
def extract_location(data_dict):
    result = {}
    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        result[split] = {
            'case_ids': merged['case_id'].values,
            'features': merged['Location_enc'].values.astype(int)
        }
    return result

loc_data = extract_location(data)

# ---- 3d. Image (per-modality concatenation, NO mean-pool) ----
def extract_image_per_modality(data_dict):
    """
    Extract per-modality image features by concatenation (NOT mean-pool).
    data_loader already applied per-modality PCA(pca_dim=128), so each
    modality is 128d. We just concatenate all 4 modalities.
    Result: 128d × 4 modalities = 512d per sample.

    WHY NOT mean-pool: T1/T1c capture anatomy+vascularity; T2/T2F capture edema.
    Mean-pooling destroys this complementary information.
    """
    result = {}
    for split in ['train', 'val', 'test']:
        img_dict = data_dict[split]['image']
        case_ids = data_dict[split]['merged']['case_id'].values
        # Get dimension from first sample's first modality
        sample_mod = next(iter(img_dict.values()))
        mod_dim = sample_mod[MODALITIES[0]].shape[0]  # 128 (already PCA-reduced by data_loader)

        X = np.zeros((len(case_ids), mod_dim * len(MODALITIES)), dtype=np.float32)
        for i, cid in enumerate(case_ids):
            cid_feats = img_dict.get(cid, {})
            for j, mod in enumerate(MODALITIES):
                feat = cid_feats.get(mod, np.zeros(mod_dim, dtype=np.float32))
                X[i, j*mod_dim:(j+1)*mod_dim] = feat
        result[split] = {'case_ids': case_ids, 'features': X}

    print(f"  Image per-mod concat: {mod_dim}d × {len(MODALITIES)} modalities = {mod_dim * len(MODALITIES)}d")
    return result

img_data = extract_image_per_modality(data)
print(f"  Image shape: {img_data['train']['features'].shape}")

# ---- 3e. Reports ----
def extract_reports(data_dict):
    result = {}
    for split in ['train', 'val', 'test']:
        report_df = data_dict[split]['report'].copy()
        report_df['case_id'] = report_df['case_id'].astype(int)
        report_df['report'] = report_df['report'].fillna('')
        merged = data_dict[split]['merged']
        case_ids = merged['case_id'].values
        cid_to_text = dict(zip(report_df['case_id'], report_df['report']))
        texts = [cid_to_text.get(cid, '') for cid in case_ids]
        result[split] = {'case_ids': case_ids, 'texts': texts}
    return result

rep_data = extract_reports(data)

print("Feature extraction complete:")
for mod in MODALITIES:
    print(f"  Radiomics {mod}: {rad_data['train'][mod].shape}")
print(f"  Clinical: {clin_data['train']['features'].shape}")
print(f"  Location: cardinality={loc_data['train']['features'].max()+1}")

print(f"  Reports: {len(rep_data['train']['texts'])} texts")


  Image per-mod concat: 128d × 4 modalities = 512d
  Image shape: (1983, 512)
Feature extraction complete:
  Radiomics ax_t1: (1983, 6)
  Radiomics ax_t1c: (1983, 6)
  Radiomics ax_t2: (1983, 6)
  Radiomics ax_t2f: (1983, 6)
  Clinical: (1983, 27)
  Location: cardinality=635
  Reports: 1983 texts


In [ ]:

# ============================================================
# QUICK DIAGNOSTIC: Which data source is most predictive?
# ============================================================

from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

print("="*70)
print("QUICK DIAGNOSTIC: Single-source baseline (RF)")
print("="*70)

# Helper
def quick_cv(X, y, name):
    clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
    print(f"  {name:<30s}: {scores.mean():.4f} (+/- {scores.std():.4f})")
    return scores.mean()

results = {}

# 1. Radiomics (all 4 combined)
X_rad = np.hstack([np.vstack([rad_data['train'][m], rad_data['val'][m]]) for m in MODALITIES])
results['Radiomics (20d)'] = quick_cv(X_rad, y_tv, 'Radiomics (all 4, 20d)')

# 2. Clinical
X_clin = np.vstack([clin_data['train']['features'], clin_data['val']['features']])
results['Clinical'] = quick_cv(X_clin, y_tv, 'Clinical')

# 3. Image (128-d pooled)
X_img = np.vstack([img_data['train']['features'], img_data['val']['features']])
results['Image (128d pooled)'] = quick_cv(X_img, y_tv, 'Image (128d pooled)')

# 4. Location
X_loc = np.concatenate([loc_data['train']['features'], loc_data['val']['features']]).reshape(-1, 1)
results['Location'] = quick_cv(X_loc, y_tv, 'Location (1d)')

# 5. Combined: Clinical + Radiomics
X_clin_rad = np.hstack([X_clin, X_rad])
results['Clinical+Radiomics'] = quick_cv(X_clin_rad, y_tv, 'Clinical + Radiomics')

# 6. Combined: All tabular
X_all_tab = np.hstack([X_clin, X_rad, X_loc])
results['All tabular'] = quick_cv(X_all_tab, y_tv, 'All tabular (no image)')

print("\n" + "="*70)
print("RANKING:")
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {score:.4f} - {name}")


QUICK DIAGNOSTIC: Single-source baseline (RF)
  Radiomics (all 4, 20d)        : 0.5221 (+/- 0.0182)
  Clinical                      : 0.6651 (+/- 0.0216)
  Image (128d pooled)           : 0.5697 (+/- 0.0128)
  Location (1d)                 : 0.6518 (+/- 0.0142)
  Clinical + Radiomics          : 0.6593 (+/- 0.0175)
  All tabular (no image)        : 0.6792 (+/- 0.0162)

RANKING:
  0.6792 - All tabular
  0.6651 - Clinical
  0.6593 - Clinical+Radiomics
  0.6518 - Location
  0.5697 - Image (128d pooled)
  0.5221 - Radiomics (20d)


In [ ]:

def cv_evaluate(name, X, y, models, n_splits=5, random_state=42):
    """
    models: list of (model_name, clf_factory, needs_scaling)
    Returns: dict of model_name -> {micro, macro, oof_probs}
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    results = {}

    for model_name, clf_factory, needs_scale in models:
        micro_s, macro_s = [], []
        oof_probs = np.zeros((len(y), NUM_CLASSES))

        for tr_idx, va_idx in skf.split(X, y):
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            if needs_scale:
                scaler = StandardScaler()
                X_tr = scaler.fit_transform(X_tr)
                X_va = scaler.transform(X_va)

            model = clf_factory()
            model.fit(X_tr, y_tr)

            if hasattr(model, 'predict_proba'):
                probs = model.predict_proba(X_va)
            else:
                preds_onehot = model.predict(X_va)
                probs = np.zeros((len(y_va), NUM_CLASSES))
                for i, p in enumerate(preds_onehot):
                    probs[i, int(p)] = 1.0

            oof_probs[va_idx] = probs
            preds = probs.argmax(axis=1)
            micro_s.append(f1_score(y_va, preds, average='micro'))
            macro_s.append(f1_score(y_va, preds, average='macro'))

        oof_preds = oof_probs.argmax(axis=1)
        full_micro = f1_score(y, oof_preds, average='micro')
        full_macro = f1_score(y, oof_preds, average='macro')
        full_wtd   = f1_score(y, oof_preds, average='weighted')

        print(f"  {name:25s} / {model_name:20s}: Micro={full_micro:.4f}, Macro={full_macro:.4f}, Wtd={full_wtd:.4f}")

        results[model_name] = {
            'micro': full_micro, 'macro': full_macro, 'weighted': full_wtd,
            'micro_std': np.std(micro_s), 'macro_std': np.std(macro_s),
            'oof_probs': oof_probs
        }

    return results


In [ ]:
# ============================================================
# MINIMAL MODEL SET - Only strongest configurations
# ============================================================

def get_models_radiomics():
    """Only XGBoost and LightGBM (strongest for tabular)."""
    return [
        ('XGB-d6-lr05', lambda: xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=3.0, objective='multi:softprob', num_class=NUM_CLASSES, eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1), False),
        ('LGB-d6-lr05', lambda: lgb.LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, num_leaves=31, subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=3.0, random_state=42, verbose=-1, n_jobs=-1) if HAS_LGB else None, False),
    ]

def get_models_clinical():
    return get_models_radiomics()

def get_models_image():
    """Only XGBoost and LightGBM for image features."""
    return [
        ('XGB-d6-lr03', lambda: xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5, reg_alpha=2.0, reg_lambda=10.0, objective='multi:softprob', num_class=NUM_CLASSES, eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1), False),
        ('LGB-d6-lr02', lambda: lgb.LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.02, num_leaves=31, subsample=0.8, colsample_bytree=0.5, reg_alpha=2.0, reg_lambda=10.0, random_state=42, verbose=-1, n_jobs=-1) if HAS_LGB else None, False),
    ]

def get_models_report():
    """Only strongest for report (SVM + XGB + LGB)."""
    return [
        ('SVM-RBF-C10', lambda: SVC(C=10, kernel='rbf', gamma='scale', probability=True, random_state=42), True),
        ('SVM-RBF-C100', lambda: SVC(C=100, kernel='rbf', gamma='scale', probability=True, random_state=42), True),
        ('LGB-d4-lr05', lambda: lgb.LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, num_leaves=15, subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=5.0, random_state=42, verbose=-1, n_jobs=-1) if HAS_LGB else None, False),
        ('XGB-d5-lr03', lambda: xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, reg_alpha=2.0, reg_lambda=10.0, objective='multi:softprob', num_class=NUM_CLASSES, eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1), False),
    ]

def get_models_location():
    """Location works well with tree models."""
    return [
        ('RF-d20', lambda: RandomForestClassifier(n_estimators=500, max_depth=20, min_samples_leaf=2, random_state=42, n_jobs=-1), False),
        ('XGB-d6', lambda: xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, verbosity=0, n_jobs=-1), False),
    ]

# Filter out None models (when libraries not installed)
def filter_models(models):
    return [(n, f, s) for n, f, s in models if f is not None]

# Override to auto-filter
_original_get_models = {}
for name in ['get_models_radiomics', 'get_models_clinical', 'get_models_image', 'get_models_report', 'get_models_location']:
    if name in dir():
        _original_get_models[name] = locals()[name]

def get_models_radiomics(): return filter_models(_original_get_models.get('get_models_radiomics', lambda: [])())
def get_models_clinical(): return filter_models(_original_get_models.get('get_models_clinical', lambda: [])())
def get_models_image(): return filter_models(_original_get_models.get('get_models_image', lambda: [])())
def get_models_report(): return filter_models(_original_get_models.get('get_models_report', lambda: [])())
def get_models_location(): return filter_models(_original_get_models.get('get_models_location', lambda: [])())


## 2. Radiomics Experiments

In [ ]:
# ============================================================
# RADIOMICS: GPU-Accelerated Training + SVM Kernel Trick
# ============================================================

print("="*70)
print("RADIOMICS: GPU + SVM KERNEL")
print("="*70)

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import cross_val_score

# Get all radiomics (4 modalities combined)
X_rad = np.hstack([np.vstack([rad_data['train'][m], rad_data['val'][m]]) for m in MODALITIES])
print(f"Base radiomics: {X_rad.shape}")

# Per-modality features (without missing flag)
def get_mod_feats(X, mod_idx):
    return X[:, mod_idx*6:mod_idx*6+5]

t1  = get_mod_feats(X_rad, 0)
t1c = get_mod_feats(X_rad, 1)
t2  = get_mod_feats(X_rad, 2)
t2f = get_mod_feats(X_rad, 3)

# Cross-modal ratios (physically meaningful for MRI)
eps = 1e-8
ratios = np.hstack([
    (t1c + eps) / (t1 + eps),   # contrast enhancement
    (t2f + eps) / (t2 + eps),   # edema pattern
    (t2 + eps) / (t1 + eps),    # tissue contrast
    t1c - t1,                    # enhancement delta
    t2f - t2,                    # edema delta
])

X_rad_feat = np.hstack([X_rad, ratios])
print(f"With ratios: {X_rad_feat.shape}")
'''
# ---- SVM with Polynomial Kernel ----
# Degree-2 poly kernel automatically generates all pairwise feature products
# (x1*x2, x1^2, x2^2, ...) — no need to manually engineer them
print("\n--- SVM Polynomial Kernel (auto feature crossing) ---")
for degree in [2, 3, 4]:
    for C in [1, 10, 100]:
        clf = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel='poly', degree=degree, C=C, coef0=1,
                        probability=True, random_state=42))
        ])
        scores = cross_val_score(clf, X_rad_feat, y_tv, cv=5, scoring='accuracy')
        if scores.mean() > 0.55:
            print(f"  poly-d{degree} C={C:>3d}: Micro={scores.mean():.4f}")

# ---- SVM with RBF Kernel ----
print("\n--- SVM RBF Kernel ---")
for C in [1, 10, 100, 1000]:
    for gamma in ['scale', 0.01, 0.1, 1.0]:
        clf = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel='rbf', C=C, gamma=gamma,
                        probability=True, random_state=42))
        ])
        scores = cross_val_score(clf, X_rad_feat, y_tv, cv=5, scoring='accuracy')
        if scores.mean() > 0.55:
            print(f"  rbf C={C:>4d} gamma={str(gamma):>5s}: Micro={scores.mean():.4f}")
'''
# ---- GPU-accelerated XGBoost ----
print("\n--- GPU-Accelerated XGBoost ---")
try:
    # Test if GPU is available for XGBoost
    test_clf = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=10)
    test_clf.fit(X_rad_feat[:10], y_tv[:10])
    gpu_available = True
    print("  GPU available for XGBoost")
except:
    gpu_available = False
    print("  GPU not available, using CPU")

tree_method = 'hist' if gpu_available else 'hist'
device = 'cuda' if gpu_available else 'cpu'

# Data augmentation + GPU XGBoost
def augment_radiomics(X, y, factor=5, noise_std=0.05):
    X_list, y_list = [X], [y]
    # Gaussian noise
    for _ in range(factor):
        X_list.append(X + np.random.randn(*X.shape).astype(np.float32) * noise_std)
        y_list.append(y)
    # Same-class mixup
    for cls in np.unique(y):
        mask = y == cls
        X_cls = X[mask]
        n = len(X_cls)
        if n < 2: continue
        for _ in range(n):
            i, j = np.random.choice(n, 2, replace=False)
            lam = np.random.beta(0.3, 0.3)
            X_list.append((lam * X_cls[i] + (1-lam) * X_cls[j]).reshape(1, -1))
            y_list.append(np.array([cls]))
    # SMOTE
    try:
        from imblearn.over_sampling import SMOTE
        sm = SMOTE(random_state=42, k_neighbors=min(3, min((y==c).sum() for c in np.unique(y))-1))
        X_s, y_s = sm.fit_resample(X, y)
        X_list.append(X_s[len(X):])
        y_list.append(y_s[len(y):])
    except: pass
    return np.vstack(X_list).astype(np.float32), np.concatenate(y_list).astype(int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for aug_name, factor, noise in [('No augment', 0, 0), ('Augment 3x', 3, 0.03), ('Augment 5x', 5, 0.05)]:
    scores = []
    for tr_idx, va_idx in skf.split(X_rad_feat, y_tv):
        X_tr, X_va = X_rad_feat[tr_idx], X_rad_feat[va_idx]
        y_tr, y_va = y_tv[tr_idx], y_tv[va_idx]
        if factor > 0:
            X_tr, y_tr = augment_radiomics(X_tr, y_tr, factor=factor, noise_std=noise)
        clf = xgb.XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.03,
            tree_method=tree_method, device=device,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=1.0, reg_lambda=3.0,
            objective='multi:softprob', num_class=NUM_CLASSES,
            eval_metric='mlogloss', random_state=42, verbosity=0
        )
        clf.fit(X_tr, y_tr)
        scores.append(f1_score(y_va, clf.predict(X_va), average='micro'))
    print(f"  XGB+GPU {aug_name:12s}: Micro={np.mean(scores):.4f} (+/- {np.std(scores):.4f})  train_size={len(X_tr)}")

# ---- GPU LightGBM ----
if HAS_LGB:
    print("\n--- GPU-Accelerated LightGBM ---")
    for aug_name, factor, noise in [('No augment', 0, 0), ('Augment 5x', 5, 0.05)]:
        scores = []
        for tr_idx, va_idx in skf.split(X_rad_feat, y_tv):
            X_tr, X_va = X_rad_feat[tr_idx], X_rad_feat[va_idx]
            y_tr, y_va = y_tv[tr_idx], y_tv[va_idx]
            if factor > 0:
                X_tr, y_tr = augment_radiomics(X_tr, y_tr, factor=factor, noise_std=noise)
            clf = lgb.LGBMClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.03,
                num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                reg_alpha=1.0, reg_lambda=3.0,
                device='gpu' if gpu_available else 'cpu',
                random_state=42, verbose=-1, n_jobs=-1
            )
            clf.fit(X_tr, y_tr)
            scores.append(f1_score(y_va, clf.predict(X_va), average='micro'))
        print(f"  LGB+GPU {aug_name:12s}: Micro={np.mean(scores):.4f} (+/- {np.std(scores):.4f})")


RADIOMICS: GPU + SVM KERNEL
Base radiomics: (2266, 24)
With ratios: (2266, 49)

--- GPU-Accelerated XGBoost ---
  GPU not available, using CPU
  XGB+GPU No augment  : Micro=0.5230 (+/- 0.0139)  train_size=1813
  XGB+GPU Augment 3x  : Micro=0.4837 (+/- 0.0111)  train_size=11477
  XGB+GPU Augment 5x  : Micro=0.4890 (+/- 0.0227)  train_size=15103

--- GPU-Accelerated LightGBM ---
  LGB+GPU No augment  : Micro=0.5150 (+/- 0.0218)
  LGB+GPU Augment 5x  : Micro=0.4872 (+/- 0.0212)


In [ ]:

print("="*70)
print("RADIOMICS EXPERIMENTS")
print("="*70)

rad_all_results = {}   # key: f'{mod}_{model_name}' -> result dict
rad_oof = {}            # key: f'{mod}_{model_name}' -> oof_probs

for mod in MODALITIES:
    print(f"\n--- {mod.upper()} (shape: {rad_data['train'][mod].shape}) ---")
    X = np.vstack([rad_data['train'][mod], rad_data['val'][mod]])
    models = [(n, f, s) for n, f, s in get_models_radiomics() if f() is not None]
    results = cv_evaluate(f'rad_{mod}', X, y_tv, models)
    for name, res in results.items():
        rad_all_results[f'{mod}_{name}'] = res
        rad_oof[f'{mod}_{name}'] = res['oof_probs']

# Per-modality top-3 ensemble
print("\n--- Per-Modality ENSEMBLE (top-3) ---")
rad_ens_oof = {}
for mod in MODALITIES:
    mod_keys = [k for k in rad_oof if k.startswith(f'{mod}_')]
    top3 = sorted(mod_keys, key=lambda k: -rad_all_results[k]['micro'])[:3]
    ens = np.mean([rad_oof[k] for k in top3], axis=0)
    preds = ens.argmax(axis=1)
    print(f"  {mod}: top3={[k.split('_', 1)[1] for k in top3]}")
    print(f"    ENSEMBLE: Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")
    rad_ens_oof[mod] = ens


RADIOMICS EXPERIMENTS

--- AX_T1 (shape: (1983, 6)) ---
  rad_ax_t1                 / XGB-d6-lr05         : Micro=0.4338, Macro=0.2056, Wtd=0.4046
  rad_ax_t1                 / LGB-d6-lr05         : Micro=0.4417, Macro=0.2088, Wtd=0.4126

--- AX_T1C (shape: (1983, 6)) ---
  rad_ax_t1c                / XGB-d6-lr05         : Micro=0.4192, Macro=0.1928, Wtd=0.3814
  rad_ax_t1c                / LGB-d6-lr05         : Micro=0.4144, Macro=0.1908, Wtd=0.3793

--- AX_T2 (shape: (1983, 6)) ---
  rad_ax_t2                 / XGB-d6-lr05         : Micro=0.4197, Macro=0.2023, Wtd=0.3854
  rad_ax_t2                 / LGB-d6-lr05         : Micro=0.4175, Macro=0.2000, Wtd=0.3842

--- AX_T2F (shape: (1983, 6)) ---
  rad_ax_t2f                / XGB-d6-lr05         : Micro=0.4612, Macro=0.2172, Wtd=0.4322
  rad_ax_t2f                / LGB-d6-lr05         : Micro=0.4554, Macro=0.2212, Wtd=0.4296

--- Per-Modality ENSEMBLE (top-3) ---
  ax_t1: top3=['t1_LGB-d6-lr05', 't1_XGB-d6-lr05']
    ENSEMBLE: Micro=0.

In [ ]:

# Radiomics ALL 4 combined (20 features)
X_rad_all = np.hstack([np.vstack([rad_data['train'][m], rad_data['val'][m]]) for m in MODALITIES])
print(f"\n--- Radiomics ALL 4 combined: {X_rad_all.shape} ---")

models = [(n, f, s) for n, f, s in get_models_radiomics() if f() is not None]
results = cv_evaluate('rad_all', X_rad_all, y_tv, models)
for name, res in results.items():
    rad_all_results[f'radall_{name}'] = res

sorted_models = sorted(results.items(), key=lambda x: -x[1]['micro'])
top3 = [m[0] for m in sorted_models[:3]]
ens = np.mean([results[n]['oof_probs'] for n in top3], axis=0)
preds = ens.argmax(axis=1)
print(f"\n  rad_all TOP-3: {top3}")
print(f"    Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")
rad_ens_oof['radall'] = ens



--- Radiomics ALL 4 combined: (2266, 24) ---
  rad_all                   / XGB-d6-lr05         : Micro=0.5093, Macro=0.2506, Wtd=0.4801
  rad_all                   / LGB-d6-lr05         : Micro=0.5040, Macro=0.2507, Wtd=0.4784

  rad_all TOP-3: ['XGB-d6-lr05', 'LGB-d6-lr05']
    Micro=0.5066, Macro=0.2473


## 3. Clinical Experiments

In [ ]:

# ============================================================
# FEATURE ENGINEERING: Clinical + Cross-Modal Ratios
# ============================================================

print("="*70)
print("FEATURE ENGINEERING EXPERIMENTS")
print("="*70)

# 1. Location One-Hot (636 dims) vs Label Encoding (1 dim)
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score
loc_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_loc_oh = loc_enc.fit_transform(np.concatenate([loc_data['train']['features'], loc_data['val']['features']]).reshape(-1, 1))
print(f"Location One-Hot: {X_loc_oh.shape}")

# 2. SI Combination Features
merged = pd.concat([data['train']['merged'], data['val']['merged']])
si_cols = [c for c in merged.columns if c.startswith('Signal Intensity') and not '_' in c.replace('Signal Intensity', '')]
print(f"SI raw columns: {si_cols}")

# Create SI combination string
si_combo = merged[si_cols].apply(lambda row: '_'.join(row.fillna('unknown').values), axis=1)
from sklearn.preprocessing import LabelEncoder
si_le = LabelEncoder()
X_si_combo = si_le.fit_transform(si_combo).reshape(-1, 1)
print(f"SI Combination: {len(np.unique(X_si_combo))} unique patterns")

# 3. Cross-Modal Radiomics Ratios (T1c/T1, FLAIR/T2)
X_rad = np.hstack([np.vstack([rad_data['train'][m], rad_data['val'][m]]) for m in MODALITIES])
# Extract original 5 features per mod (without missing flag)
def get_rad_features(mod_idx):
    start = mod_idx * 6
    return X_rad[:, start:start+5]

t1 = get_rad_features(0)
t1c = get_rad_features(1)
t2 = get_rad_features(2)
t2f = get_rad_features(3)

# Ratios (add small epsilon to avoid div by zero)
eps = 1e-8
ratios = np.hstack([
    (t1c + eps) / (t1 + eps),  # T1c/T1: contrast enhancement
    (t2f + eps) / (t2 + eps),  # FLAIR/T2: edema pattern
])
print(f"Cross-modal ratios: {ratios.shape}")

# 4. Combine all engineered features
X_clin = np.vstack([clin_data['train']['features'], clin_data['val']['features']])
X_eng = np.hstack([X_clin, X_loc_oh, ratios])
print(f"\nEngineered features total: {X_eng.shape}")

# Quick CV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb

for name, clf in [
    ('RF-500', RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=3, random_state=42, n_jobs=-1)),
    ('GB-200', GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)),
    ('XGB-d6', xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.03, random_state=42, n_jobs=-1)),
]:
    scores = cross_val_score(clf, X_eng, y_tv, cv=5, scoring='accuracy', n_jobs=-1)
    print(f"  {name:10s}: {scores.mean():.4f} (+/- {scores.std():.4f})")


FEATURE ENGINEERING EXPERIMENTS
Location One-Hot: (2266, 572)
SI raw columns: []
SI Combination: 1 unique patterns
Cross-modal ratios: (2266, 10)

Engineered features total: (2266, 609)
  RF-500    : 0.6576 (+/- 0.0251)
  GB-200    : 0.7498 (+/- 0.0073)
  XGB-d6    : 0.7087 (+/- 0.0127)


In [ ]:

# ============================================================
# GRADIENT BOOSTING TUNING (Best from feature engineering)
# ============================================================

print("="*70)
print("GRADIENT BOOSTING TUNING")
print("="*70)

from sklearn.ensemble import GradientBoostingClassifier

# Grid search key hyperparameters
best_score = 0
best_params = None

for n_est in [500, 700]:
    for depth in [10, 12, 15]:
        for lr in [0.2]:
            for subsample in [1.0]:
                clf = GradientBoostingClassifier(
                    n_estimators=n_est, max_depth=depth, learning_rate=lr,
                    subsample=subsample, random_state=42
                )
                scores = cross_val_score(clf, X_eng, y_tv, cv=5, scoring='accuracy')
                mean_score = scores.mean()
                if mean_score > best_score:
                    best_score = mean_score
                    best_params = (n_est, depth, lr, subsample)
                    print(f"  NEW BEST: n_est={n_est}, depth={depth}, lr={lr}, subsample={subsample}: {mean_score:.4f}")

print(f"\nBEST: {best_score:.4f} with params {best_params}")


GRADIENT BOOSTING TUNING
  NEW BEST: n_est=500, depth=10, lr=0.2, subsample=1.0: 0.7705
  NEW BEST: n_est=700, depth=10, lr=0.2, subsample=1.0: 0.7719

BEST: 0.7719 with params (700, 10, 0.2, 1.0)


In [ ]:

print("\n" + "="*70)
print("CLINICAL EXPERIMENTS")
print("="*70)

X_clin = np.vstack([clin_data['train']['features'], clin_data['val']['features']])
print(f"Clinical features: {X_clin.shape}")

models = [(n, f, s) for n, f, s in get_models_radiomics() if f() is not None]
results = cv_evaluate('clinical', X_clin, y_tv, models)
clin_results = results

sorted_models = sorted(results.items(), key=lambda x: -x[1]['micro'])
top3 = [m[0] for m in sorted_models[:3]]
clin_ens_oof = np.mean([results[n]['oof_probs'] for n in top3], axis=0)
preds = clin_ens_oof.argmax(axis=1)
print(f"\n  clinical TOP-3: {top3}")
print(f"    Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")



CLINICAL EXPERIMENTS
Clinical features: (2266, 27)
  clinical                  / XGB-d6-lr05         : Micro=0.6598, Macro=0.4123, Wtd=0.6393
  clinical                  / LGB-d6-lr05         : Micro=0.6531, Macro=0.3773, Wtd=0.6241

  clinical TOP-3: ['XGB-d6-lr05', 'LGB-d6-lr05']
    Micro=0.6602, Macro=0.3991


## 4. Image Experiments

In [ ]:
# ============================================================
# IMAGE + RADIOMICS FEATURE ENGINEERING COMBINED
# Key insight: Image (ResNet per-mod PCA) + Radiomics cross-modal
# ratios should be concatenated (not stacked/averaged) since they
# capture complementary info at different scales.
# ============================================================

print("="*70)
print("IMAGE + RADIOMICS FEATURE ENGINEERING")
print("="*70)

# ---- 1. Radiomics Cross-Modal Feature Engineering ----
# Radiomics has only 5 features per modality × 4 modalities = 20 raw
# The key medical signals come from CROSS-MODAL RATIOS:
#   T1c/T1 = enhancement (BBB disruption, tumor vascularity)
#   T2F/T2 = edema pattern (vasogenic vs infiltrative)
#   T1/T2  = tissue contrast (fat, blood, calcification)
eps = 1e-8

def extract_radiomics_engineered(data_dict):
    """Extract radiomics with cross-modal ratios — medically meaningful."""
    result = {}
    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        case_ids = merged['case_id'].values

        # Raw radiomics: 5 features × 4 modalities = 20 features
        raw_feats = []
        for mod in MODALITIES:
            cols = [f'{mod}_{feat}' for feat in RAD_FEAT_NAMES]
            vals = merged[cols].fillna(0).values.astype(np.float32)
            raw_feats.append(vals)
        raw = np.hstack(raw_feats)  # (N, 20)

        # Extract per-modality arrays for ratio computation
        # Each block: 5 features
        mod_data = {
            'ax_t1':  raw[:, 0:5],
            'ax_t1c': raw[:, 5:10],
            'ax_t2':  raw[:, 10:15],
            'ax_t2f': raw[:, 15:20],
        }

        # Cross-modal ratios: for each of the 5 radiomics features,
        # compute medically meaningful ratios
        ratios = []
        for i in range(5):
            # Enhancement: T1c/T1 (contrast uptake relative to pre-contrast)
            enh = (mod_data['ax_t1c'][:, i] + eps) / (mod_data['ax_t1'][:, i] + eps)
            # Edema: T2F/T2 (fluid content relative to proton density)
            edm = (mod_data['ax_t2f'][:, i] + eps) / (mod_data['ax_t2'][:, i] + eps)
            # Tissue contrast: T1/T2 (relative to T2 signal)
            tc  = (mod_data['ax_t1'][:, i] + eps) / (mod_data['ax_t2'][:, i] + eps)
            # Delta enhancement: T1c - T1 (absolute enhancement)
            den = mod_data['ax_t1c'][:, i] - mod_data['ax_t1'][:, i]
            # Delta edema: T2F - T2 (absolute edema)
            ded = mod_data['ax_t2f'][:, i] - mod_data['ax_t2'][:, i]
            ratios.extend([enh, edm, tc, den, ded])

        ratios = np.column_stack(ratios)  # (N, 25)

        # Clip extreme ratios
        ratios = np.clip(ratios, 0.01, 100.0)

        # Log-transform ratios (medical features are typically log-normal)
        ratios_log = np.log(ratios)

        # Missing flags per modality
        missing = []
        for mod in MODALITIES:
            cols = [f'{mod}_{feat}' for feat in RAD_FEAT_NAMES]
            m = merged.get(f'{mod}_missing', pd.Series(0, index=merged.index)).values.reshape(-1, 1)
            missing.append(m)
        missing_flags = np.hstack(missing)  # (N, 4)

        # Final: raw (20) + log-ratios (25) + missing (4) = 49 features
        X = np.hstack([raw, ratios_log, missing_flags])
        result[split] = {'case_ids': case_ids, 'features': X}

    return result

rad_eng_data = extract_radiomics_engineered(data)

# Raw radiomics (for comparison)
X_rad = np.hstack([np.vstack([rad_data['train'][m], rad_data['val'][m]]) for m in MODALITIES])
X_rad_eng = np.vstack([rad_eng_data['train']['features'], rad_eng_data['val']['features']])
print(f"Radiomics engineered: {X_rad_eng.shape} (raw 20 + log-ratios 25 + missing 4)")

# ---- 2. Image per-modality PCA features ----
# img_data already has per-mod PCA from Cell 4
X_img = np.vstack([img_data['train']['features'], img_data['val']['features']])
print(f"Image per-mod concat: {X_img.shape} (128d × 4 modalities = 512d)")

# ---- 3. Concatenate Image + Radiomics ----
X_img_rad = np.hstack([X_img, X_rad_eng])
print(f"Image + Radiomics concat: {X_img_rad.shape}")

# ---- 4. Quick CV comparison ----
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

for name, X in [
    ('Radiomics raw (20d)',  X_rad),
    ('Radiomics eng (49d)',  X_rad_eng),
    ('Image concat (512d)', X_img),
    ('Image+Rad eng (561d)', X_img_rad),
]:
    clf = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.5,
        reg_alpha=1.0, reg_lambda=5.0,
        objective='multi:softprob', num_class=NUM_CLASSES,
        eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1
    )
    scores = cross_val_score(clf, X, y_tv, cv=5, scoring='f1_micro', n_jobs=-1)
    print(f"  {name:<25s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

# ---- 5. Full CV evaluation with all models ----
print(f"\n--- Image + Radiomics FULL CV ---")
models = [(n, f, s) for n, f, s in get_models_radiomics() if f() is not None]
img_rad_results = cv_evaluate('img_rad', X_img_rad, y_tv, models)

# ---- 6. Top-3 ensemble ----
sorted_models = sorted(img_rad_results.items(), key=lambda x: -x[1]['micro'])
top3 = [m[0] for m in sorted_models[:3]]
img_rad_ens_oof = np.mean([img_rad_results[n]['oof_probs'] for n in top3], axis=0)
preds = img_rad_ens_oof.argmax(axis=1)
print(f"\n  Image+Rad TOP-3: {top3}")
print(f"    Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")



IMAGE + RADIOMICS FEATURE ENGINEERING
Radiomics engineered: (2266, 49) (raw 20 + log-ratios 25 + missing 4)
Image per-mod concat: (2266, 512) (128d × 4 modalities = 512d)
Image + Radiomics concat: (2266, 561)
  Radiomics raw (20d)      : 0.5168 (+/- 0.0133)
  Radiomics eng (49d)      : 0.5132 (+/- 0.0214)
  Image concat (512d)      : 0.5966 (+/- 0.0122)
  Image+Rad eng (561d)     : 0.6112 (+/- 0.0156)

--- Image + Radiomics FULL CV ---
  img_rad                   / XGB-d6-lr05         : Micro=0.5997, Macro=0.2877, Wtd=0.5581
  img_rad                   / LGB-d6-lr05         : Micro=0.6046, Macro=0.2933, Wtd=0.5656

  Image+Rad TOP-3: ['LGB-d6-lr05', 'XGB-d6-lr05']
    Micro=0.6117, Macro=0.2978


In [ ]:

print("\n" + "="*70)
print("IMAGE (per-mod concat 128d×4=512d, for reference only)")
print("  Note: Image+Rad combined is in the previous cell. This cell shows image-only baseline.")
print("="*70)

X_img = np.vstack([img_data['train']['features'], img_data['val']['features']])
print(f"Image features: {X_img.shape}")

models = [(n, f, s) for n, f, s in get_models_image() if f() is not None]
results = cv_evaluate('image', X_img, y_tv, models)
img_results = results

sorted_models = sorted(results.items(), key=lambda x: -x[1]['micro'])
top3 = [m[0] for m in sorted_models[:3]]
img_ens_oof = np.mean([results[n]['oof_probs'] for n in top3], axis=0)
preds = img_ens_oof.argmax(axis=1)
print(f"\n  image TOP-3: {top3}")
print(f"    Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")



IMAGE (per-mod concat 128d×4=512d, for reference only)
  Note: Image+Rad combined is in the previous cell. This cell shows image-only baseline.
Image features: (2266, 512)
  image                     / XGB-d6-lr03         : Micro=0.5936, Macro=0.2702, Wtd=0.5442
  image                     / LGB-d6-lr02         : Micro=0.5891, Macro=0.2763, Wtd=0.5450

  image TOP-3: ['XGB-d6-lr03', 'LGB-d6-lr02']
    Micro=0.5931, Macro=0.2738


In [ ]:
# ============================================================
# GRADIENT BOOSTED TREE COMPARISON: CatBoost vs XGBoost vs LightGBM
# Focus: Radiomics (with ratios) and Image features
# ============================================================

print("="*70)
print("GRADIENT BOOSTED TREE COMPARISON")
print("CatBoost vs XGBoost vs LightGBM")
print("="*70)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
    HAS_LGB = True
    print("LightGBM: Available")
except ImportError:
    HAS_LGB = False
    print("LightGBM: Not available")

try:
    from catboost import CatBoostClassifier
    HAS_CB = True
    print("CatBoost: Available")
except ImportError:
    HAS_CB = False
    print("CatBoost: Not available")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# ============================================================
# HELPER: Build Radiomics features with cross-modal ratios
# ============================================================
def build_radiomics_features(data_dict, use_ratios=True):
    """Build radiomics features from merged data."""
    MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']
    RAD_FEAT_NAMES = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
                      'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']
    eps = 1e-8
    result = {}

    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        case_ids = merged['case_id'].values

        # Raw radiomics: 5 features x 4 modalities = 20d
        raw_list = []
        for mod in MODALITIES:
            cols = [f'{mod}_{feat}' for feat in RAD_FEAT_NAMES]
            vals = merged[cols].fillna(0).values.astype(np.float32)
            raw_list.append(vals)
        raw = np.hstack(raw_list)

        if use_ratios:
            mod_data = {
                'ax_t1':  raw[:, 0:5],
                'ax_t1c': raw[:, 5:10],
                'ax_t2':  raw[:, 10:15],
                'ax_t2f': raw[:, 15:20],
            }

            ratios = []
            for i in range(5):
                t1  = mod_data['ax_t1'][:, i]
                t1c = mod_data['ax_t1c'][:, i]
                t2  = mod_data['ax_t2'][:, i]
                t2f = mod_data['ax_t2f'][:, i]
                ratios.append(np.log(np.clip((t1c + eps) / (t1 + eps), eps, 100)))
                ratios.append(np.log(np.clip((t2f + eps) / (t2 + eps), eps, 100)))
                ratios.append(np.log(np.clip((t1 + eps) / (t2 + eps), eps, 100)))
                ratios.append(np.clip(t1c - t1, -100, 100))
                ratios.append(np.clip(t2f - t2, -100, 100))
            ratios = np.column_stack(ratios)
            X = np.hstack([raw, ratios])
        else:
            X = raw

        missing = np.column_stack([
            merged.get(f'{mod}_missing', pd.Series(0, index=merged.index)).values.astype(np.float32)
            for mod in MODALITIES
        ])
        X = np.hstack([X, missing])
        result[split] = {'case_ids': case_ids, 'X': X}

    return result


def build_image_features(data_dict, pca_dim=128):
    """Build per-modality PCA image features."""
    MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']
    result = {}

    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        img_dict = data_dict[split]['image']
        case_ids = merged['case_id'].values

        img_list = []
        for cid in case_ids:
            cid_feats = img_dict.get(cid, {})
            feats = [cid_feats.get(mod, np.zeros(pca_dim, dtype=np.float32)) for mod in MODALITIES]
            img_list.append(np.concatenate(feats))
        X = np.stack(img_list).astype(np.float32)
        result[split] = {'case_ids': case_ids, 'X': X}

    return result


def run_cv_comparison(X, y, model_factories, model_names, skf):
    """Run 5-fold CV for multiple models."""
    N, K = len(y), NUM_CLASSES
    results = {name: {'oof': np.zeros((N, K)), 'fold_scores': []} for name in model_names}

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        for name, factory in zip(model_names, model_factories):
            model = factory()
            model.fit(X_tr, y_tr)
            probs = model.predict_proba(X_va)
            results[name]['oof'][va_idx] = probs
            f1 = f1_score(y_va, probs.argmax(axis=1), average='micro')
            results[name]['fold_scores'].append(f1)

    for name in model_names:
        results[name]['oof_f1'] = f1_score(y, results[name]['oof'].argmax(axis=1), average='micro')
        results[name]['oof_macro'] = f1_score(y, results[name]['oof'].argmax(axis=1), average='macro')

    return results


def print_results_table(results, title=""):
    if title:
        print(f"{title}")
    print(f"{'Model':<20s} | {'Micro-F1':>10s} | {'Macro-F1':>10s} | Fold scores")
    print("-" * 70)
    for name, r in results.items():
        scores = r['fold_scores']
        print(f"{name:<20s} | {r['oof_f1']:>10.4f} | {r['oof_macro']:>10.4f} | {np.mean(scores):.4f} +/- {np.std(scores):.4f}")


# ============================================================
# EXP 1: Radiomics (raw + ratios, 49d)
# ============================================================
print("" + "="*70)
print("EXPERIMENT 1: Radiomics (raw 20d + ratios 25d + missing 4d = 49d)")
print("="*70)

rad_feat = build_radiomics_features(data, use_ratios=True)
X_rad_all = np.vstack([rad_feat['train']['X'], rad_feat['val']['X']])
y_tv_arr = np.concatenate([y_train, y_val])
print(f"Radiomics features: train+val={X_rad_all.shape}")

rad_factories, rad_names = [], ['XGBoost', 'LightGBM']
rad_factories.append(lambda: xgb.XGBClassifier(
    n_estimators=1000, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=1.0, reg_lambda=3.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1))
if HAS_LGB:
    rad_factories.append(lambda: lgb.LGBMClassifier(
        n_estimators=1000, max_depth=3, learning_rate=0.05,
        num_leaves=31, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=1.0, reg_lambda=3.0,
        random_state=42, verbose=-1, n_jobs=-1))
if HAS_CB:
    rad_names.append('CatBoost')
    rad_factories.append(lambda: CatBoostClassifier(
        iterations=1000, depth=3, learning_rate=0.05,
        l2_leaf_reg=3.0, random_state=42, verbose=0, thread_count=-1))

rad_results = run_cv_comparison(X_rad_all, y_tv_arr, rad_factories, rad_names, skf)
print_results_table(rad_results, "Radiomics 5-Fold CV")

print(f"Per-class ({max(rad_results.items(), key=lambda x: x[1]['oof_f1'])[0]}):")
for i, cls in enumerate(label_encoder.classes_):
    mask = (y_tv_arr == i)
    if mask.sum() > 0:
        acc = (rad_results[max(rad_results.items(), key=lambda x: x[1]['oof_f1'])[0]]['oof'][mask].argmax(axis=1) == i).mean()
        print(f"    {str(cls)[:40]:40s}: acc={acc:.4f} (n={mask.sum()})")


# ============================================================
# EXP 2: Image (ResNet PCA, 512d)
# ============================================================
print("" + "="*70)
print("EXPERIMENT 2: Image (ResNet PCA 128d x 4 = 512d)")
print("="*70)

img_feat = build_image_features(data, pca_dim=128)
X_img_all = np.vstack([img_feat['train']['X'], img_feat['val']['X']])
print(f"Image features: {X_img_all.shape}")

img_factories, img_names = [], ['XGBoost', 'LightGBM']
img_factories.append(lambda: xgb.XGBClassifier(
    n_estimators=1000, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.5,
    reg_alpha=2.0, reg_lambda=10.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1))
if HAS_LGB:
    img_factories.append(lambda: lgb.LGBMClassifier(
        n_estimators=800, max_depth=3, learning_rate=0.03,
        num_leaves=63, subsample=0.8, colsample_bytree=0.5,
        reg_alpha=2.0, reg_lambda=10.0,
        random_state=42, verbose=-1, n_jobs=-1))
if HAS_CB:
    img_names.append('CatBoost')
    img_factories.append(lambda: CatBoostClassifier(
        iterations=800, depth=3, learning_rate=0.03,
        l2_leaf_reg=10.0, random_state=42, verbose=0, thread_count=-1))

img_results = run_cv_comparison(X_img_all, y_tv_arr, img_factories, img_names, skf)
print_results_table(img_results, "Image 5-Fold CV")

print(f"Per-class ({max(img_results.items(), key=lambda x: x[1]['oof_f1'])[0]}):")
for i, cls in enumerate(label_encoder.classes_):
    mask = (y_tv_arr == i)
    if mask.sum() > 0:
        best_name = max(img_results.items(), key=lambda x: x[1]['oof_f1'])[0]
        acc = (img_results[best_name]['oof'][mask].argmax(axis=1) == i).mean()
        print(f"    {str(cls)[:40]:40s}: acc={acc:.4f} (n={mask.sum()})")


# ============================================================
# EXP 3: Combined (Radiomics + Image, 561d)
# ============================================================
print("" + "="*70)
print("EXPERIMENT 3: Radiomics + Image (49d + 512d = 561d)")
print("="*70)

X_comb_all = np.hstack([X_rad_all, X_img_all])
print(f"Combined features: {X_comb_all.shape}")

comb_factories, comb_names = [], ['XGBoost', 'LightGBM']
comb_factories.append(lambda: xgb.XGBClassifier(
    n_estimators=1000, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.5,
    reg_alpha=2.0, reg_lambda=10.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1))
if HAS_LGB:
    comb_factories.append(lambda: lgb.LGBMClassifier(
        n_estimators=1000, max_depth=3, learning_rate=0.03,
        num_leaves=63, subsample=0.8, colsample_bytree=0.5,
        reg_alpha=2.0, reg_lambda=10.0,
        random_state=42, verbose=-1, n_jobs=-1))
if HAS_CB:
    comb_names.append('CatBoost')
    comb_factories.append(lambda: CatBoostClassifier(
        iterations=1000, depth=3, learning_rate=0.03,
        l2_leaf_reg=10.0, random_state=42, verbose=0, thread_count=-1))

comb_results = run_cv_comparison(X_comb_all, y_tv_arr, comb_factories, comb_names, skf)
print_results_table(comb_results, "Combined 5-Fold CV")

print(f"Per-class ({max(comb_results.items(), key=lambda x: x[1]['oof_f1'])[0]}):")
for i, cls in enumerate(label_encoder.classes_):
    mask = (y_tv_arr == i)
    if mask.sum() > 0:
        best_name = max(comb_results.items(), key=lambda x: x[1]['oof_f1'])[0]
        acc = (comb_results[best_name]['oof'][mask].argmax(axis=1) == i).mean()
        print(f"    {str(cls)[:40]:40s}: acc={acc:.4f} (n={mask.sum()})")


# ============================================================
# EXP 4: Ratios ONLY (no raw radiomics)
# ============================================================
print("" + "="*70)
print("EXPERIMENT 4: Cross-modal Ratios ONLY (25d + 4d = 29d)")
print("="*70)

eps = 1e-8

def build_ratios_only(data_dict):
    result = {}
    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        case_ids = merged['case_id'].values

        raw_list = []
        for mod in MODALITIES:
            cols = [f'{mod}_{feat}' for feat in RAD_FEAT_NAMES]
            vals = merged[cols].fillna(0).values.astype(np.float32)
            raw_list.append(vals)
        raw = np.hstack(raw_list)

        mod_data = {'ax_t1': raw[:, 0:5], 'ax_t1c': raw[:, 5:10],
                    'ax_t2': raw[:, 10:15], 'ax_t2f': raw[:, 15:20]}

        ratios = []
        for i in range(5):
            t1 = mod_data['ax_t1'][:, i]; t1c = mod_data['ax_t1c'][:, i]
            t2 = mod_data['ax_t2'][:, i]; t2f = mod_data['ax_t2f'][:, i]
            ratios.extend([
                np.log(np.clip((t1c+eps)/(t1+eps), eps, 100)),
                np.log(np.clip((t2f+eps)/(t2+eps), eps, 100)),
                np.log(np.clip((t1+eps)/(t2+eps), eps, 100)),
                np.clip(t1c - t1, -100, 100),
                np.clip(t2f - t2, -100, 100),
            ])
        ratios = np.column_stack(ratios)

        missing = np.column_stack([
            merged.get(f'{mod}_missing', pd.Series(0, index=merged.index)).values.astype(np.float32)
            for mod in MODALITIES
        ])

        result[split] = {'case_ids': case_ids, 'X': np.hstack([ratios, missing])}
    return result

rat_only = build_ratios_only(data)
X_ratio_all = np.vstack([rat_only['train']['X'], rat_only['val']['X']])
print(f"Ratios only: {X_ratio_all.shape}")

ratio_factories, ratio_names = [], ['XGBoost', 'LightGBM']
ratio_factories.append(lambda: xgb.XGBClassifier(
    n_estimators=1000, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.5, reg_lambda=2.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1))
if HAS_LGB:
    ratio_factories.append(lambda: lgb.LGBMClassifier(
        n_estimators=1000, max_depth=3, learning_rate=0.05,
        num_leaves=15, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, verbose=-1, n_jobs=-1))
if HAS_CB:
    ratio_names.append('CatBoost')
    ratio_factories.append(lambda: CatBoostClassifier(
        iterations=1000, depth=3, learning_rate=0.05,
        l2_leaf_reg=2.0, random_state=42, verbose=0, thread_count=-1))

ratio_results = run_cv_comparison(X_ratio_all, y_tv_arr, ratio_factories, ratio_names, skf)
print_results_table(ratio_results, "Ratios Only 5-Fold CV")


# ============================================================
# SUMMARY TABLE
# ============================================================
print("" + "="*70)
print("SUMMARY: CatBoost vs XGBoost vs LightGBM")
print("="*70)

print(f"{'Dataset':<35s} | {'XGBoost':>10s} | {'LightGBM':>10s} | {'CatBoost':>10s}")
print("-" * 70)

datasets_results = [
    ("Radiomics (raw+ratios, 49d)", rad_results),
    ("Image (PCA, 512d)", img_results),
    ("Radiomics+Image (561d)", comb_results),
    ("Ratios only (29d)", ratio_results),
]

for name, res in datasets_results:
    xgb_f1 = res.get('XGBoost', {}).get('oof_f1', float('nan'))
    lgb_f1 = res.get('LightGBM', {}).get('oof_f1', float('nan'))
    cb_f1  = res.get('CatBoost', {}).get('oof_f1', float('nan'))
    print(f"{name:<35s} | {xgb_f1:>10.4f} | {lgb_f1:>10.4f} | {cb_f1:>10.4f}")

print(f"Best per dataset:")
for name, res in datasets_results:
    best = max(res.items(), key=lambda x: x[1]['oof_f1'])
    print(f"  {name}: {best[0]} = {best[1]['oof_f1']:.4f}")

# Save OOF for later blending
gbt_xgb_oof = rad_results.get('XGBoost', {}).get('oof')
gbt_lgb_oof = rad_results.get('LightGBM', {}).get('oof')
gbt_cb_oof  = rad_results.get('CatBoost', {}).get('oof')

print("[GBC comparison cell ready]")


GRADIENT BOOSTED TREE COMPARISON
CatBoost vs XGBoost vs LightGBM
LightGBM: Available
CatBoost: Not available
EXPERIMENT 1: Radiomics (raw 20d + ratios 25d + missing 4d = 49d)
Radiomics features: train+val=(2266, 49)
Radiomics 5-Fold CV
Model                |   Micro-F1 |   Macro-F1 | Fold scores
----------------------------------------------------------------------
XGBoost              |     0.5079 |     0.2615 | 0.5080 +/- 0.0163
LightGBM             |     0.5097 |     0.2662 | 0.5097 +/- 0.0156
Per-class (LightGBM):
    Brain Metastase Tumour                  : acc=0.1285 (n=288)
    Glioma                                  : acc=0.6477 (n=1056)
    Meningioma                              : acc=0.5192 (n=832)
    Pineal tumour and Choroid plexus tumour : acc=0.0000 (n=26)
    Tumors of the sellar region             : acc=0.0312 (n=64)
EXPERIMENT 2: Image (ResNet PCA 128d x 4 = 512d)
Image features: (2266, 512)
Image 5-Fold CV
Model                |   Micro-F1 |   Macro-F1 | Fold score

In [ ]:
# ============================================================
# GBT ON REPORT + TABULAR COMBINATION
# Hypothesis: 96% groups likely used GBT (e.g. CatBoost/XGBoost) on
# Report features combined with clinical/radiomics/location features.
# Report alone via TF-IDF+SVD+LGB reaches ~85%, but combining with
# tabular features via GBT should push higher.
# ============================================================

print("="*70)
print("GBT ON REPORT + TABULAR COMBINATION")
print("Testing: Can GBT reach 96% with combined features?")
print("="*70)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack as sp_hstack, vstack as sp_vstack, issparse
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_tv_arr = np.concatenate([y_train, y_val])


# ============================================================
# Build Report features (TF-IDF)
# ============================================================
print("\n--- Building Report TF-IDF features ---")

# Get reports
def get_reports(data_dict):
    reports = {}
    for split in ['train', 'val', 'test']:
        df = data_dict[split]['merged'][['case_id']].copy()
        report_df = data_dict[split]['report']
        df = df.merge(report_df[['case_id', 'report']], on='case_id', how='left')
        df['report'] = df['report'].fillna('')
        reports[split] = dict(zip(df['case_id'], df['report']))
    return reports

reports = get_reports(data)

# Build ordered case_id list
train_cids = list(reports['train'].keys())
val_cids   = list(reports['val'].keys())
test_cids  = list(reports['test'].keys())

# Align with y_train / y_val
train_texts = [reports['train'][cid] for cid in train_cids]
val_texts   = [reports['val'][cid]   for cid in val_cids]
test_texts  = [reports['test'][cid]   for cid in test_cids]

# Fit TF-IDF on train only (prevent leakage)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                         sublinear_tf=True, min_df=2)
tfidf.fit(train_texts)

X_tfidf_train = tfidf.transform(train_texts)
X_tfidf_val   = tfidf.transform(val_texts)
X_tfidf_test  = tfidf.transform(test_texts)

# Also try with SVD for comparison
for svd_dim in [64, 128]:
    svd = TruncatedSVD(n_components=svd_dim, random_state=42)
    svd.fit(X_tfidf_train)
    X_svd_train = svd.transform(X_tfidf_train)
    X_svd_val   = svd.transform(X_tfidf_val)
    X_svd_test  = svd.transform(X_tfidf_test)
    exec(f"X_svd{svd_dim}_train = X_svd_train; X_svd{svd_dim}_val = X_svd_val; X_svd{svd_dim}_test = X_svd_test")

print(f"TF-IDF: train={X_tfidf_train.shape}, val={X_tfidf_val.shape}")
print(f"SVD-64: {X_svd64_train.shape}, SVD-128: {X_svd128_train.shape}")


# ============================================================
# Build Tabular features (Clinical + SI + Location + Radiomics)
# ============================================================
print("\n--- Building Tabular features ---")

def build_tabular_v2(data_dict):
    """Build tabular features with cross-modal ratios."""
    MODALITIES_TAB = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']
    RAD_NAMES = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
                 'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']
    eps = 1e-8
    result = {}

    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        case_ids = merged['case_id'].values

        # Clinical: Sex, Age, Age_missing (3d)
        clin = merged[['Sex_enc', 'Age_clean', 'Age_missing']].values.astype(np.float32)

        # Radiomics: raw (20d)
        raw_list = []
        for mod in MODALITIES_TAB:
            cols = [f'{mod}_{r}' for r in RAD_NAMES]
            vals = merged[cols].fillna(0).values.astype(np.float32)
            raw_list.append(vals)
        raw = np.hstack(raw_list)

        # Cross-modal ratios (25d)
        md = {'ax_t1': raw[:,0:5], 'ax_t1c': raw[:,5:10],
              'ax_t2': raw[:,10:15], 'ax_t2f': raw[:,15:20]}
        ratios = []
        for i in range(5):
            t1=md['ax_t1'][:,i]; t1c=md['ax_t1c'][:,i]
            t2=md['ax_t2'][:,i];  t2f=md['ax_t2f'][:,i]
            ratios.extend([
                np.log(np.clip((t1c+eps)/(t1+eps), eps, 100)),
                np.log(np.clip((t2f+eps)/(t2+eps), eps, 100)),
                np.log(np.clip((t1+eps)/(t2+eps), eps, 100)),
                np.clip(t1c - t1, -100, 100),
                np.clip(t2f - t2, -100, 100),
            ])
        ratios = np.column_stack(ratios)

        # Missing flags (4d)
        missing = np.column_stack([
            merged.get(f'{mod}_missing', pd.Series(0,index=merged.index)).values.astype(np.float32)
            for mod in MODALITIES_TAB])

        # SI features (24d)
        si_cols = [c for c in merged.columns if 'Signal Intensity' in c]
        si = merged[si_cols].values.astype(np.float32)

        # Location (label-encoded, 1d)
        loc_vals = merged['Location_enc'].values.astype(np.int64)

        X = np.hstack([clin, raw, ratios, missing, si, loc_vals.reshape(-1,1)])
        result[split] = {'X': X, 'case_ids': case_ids}
    return result

tab = build_tabular_v2(data)
X_tab_train = tab['train']['X']
X_tab_val   = tab['val']['X']
X_tab_test  = tab['test']['X']
print(f"Tabular features: train={X_tab_train.shape}")


# ============================================================
# EXP 1: Report TF-IDF SVD + Tabular (dense)
# ============================================================
print("\n" + "="*70)
print("EXPERIMENT 1: Report SVD-64 + Tabular (~92d)")
print("="*70)

# Align order: train/val match y_train/y_val order
X_comb_train = np.hstack([X_svd64_train, X_tab_train])
X_comb_val   = np.hstack([X_svd64_val,   X_tab_val])
X_comb_test  = np.hstack([X_svd64_test,  X_tab_test])
X_comb_all   = np.vstack([X_comb_train, X_comb_val])
print(f"Combined features: {X_comb_all.shape}")

models_comb = []
names_comb = ['XGBoost', 'LightGBM']
models_comb.append(lambda: xgb.XGBClassifier(
    n_estimators=1000, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=1.0, reg_lambda=5.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1))
if HAS_LGB:
    models_comb.append(lambda: lgb.LGBMClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.03,
        num_leaves=63, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=1.0, reg_lambda=5.0,
        random_state=42, verbose=-1, n_jobs=-1))
if HAS_CB:
    names_comb.append('CatBoost')
    models_comb.append(lambda: CatBoostClassifier(
        iterations=1000, depth=6, learning_rate=0.03,
        l2_leaf_reg=5.0, random_state=42, verbose=0, thread_count=-1))

print(f"\n{'Model':<20s} | {'Micro-F1':>10s} | {'Macro-F1':>10s}")
print("-" * 45)
for name, factory in zip(names_comb, models_comb):
    oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
    fold_scores = []
    for tr_idx, va_idx in skf.split(X_comb_all, y_tv_arr):
        m = factory()
        m.fit(X_comb_all[tr_idx], y_tv_arr[tr_idx])
        probs = m.predict_proba(X_comb_all[va_idx])
        oof[va_idx] = probs
        fold_scores.append(f1_score(y_tv_arr[va_idx], probs.argmax(axis=1), average='micro'))
    oof_f1 = f1_score(y_tv_arr, oof.argmax(axis=1), average='micro')
    oof_macro = f1_score(y_tv_arr, oof.argmax(axis=1), average='macro')
    print(f"{name:<20s} | {oof_f1:>10.4f} | {oof_macro:>10.4f}")


# ============================================================
# EXP 2: Report TF-IDF (no SVD) + Tabular via sparse matrix
# Try CatBoost on sparse features directly
# ============================================================
print("\n" + "="*70)
print("EXPERIMENT 2: Report TF-IDF (full sparse) + Tabular")
print("="*70)

# Combine as sparse matrix (full TF-IDF, no SVD)
X_sparse_tab_train = sp_hstack([X_tfidf_train, X_tab_train]).tocsr()
X_sparse_tab_val   = sp_hstack([X_tfidf_val,   X_tab_val  ]).tocsr()
X_sparse_tab_all   = sp_vstack([X_sparse_tab_train, X_sparse_tab_val]).tocsr()
print(f"Sparse combined: {X_sparse_tab_all.shape} (TF-IDF {X_tfidf_train.shape[1]} + Tab {X_tab_train.shape[1]})")

# LightGBM handles sparse natively
if HAS_LGB:
    print("\nLightGBM on full TF-IDF + Tabular (sparse):")
    oof_lgb_sparse = np.zeros((len(y_tv_arr), NUM_CLASSES))
    fold_scores = []
    for tr_idx, va_idx in skf.split(X_sparse_tab_all, y_tv_arr):
        m = lgb.LGBMClassifier(
            n_estimators=1000, max_depth=8, learning_rate=0.03,
            num_leaves=127, subsample=0.8, colsample_bytree=0.3,
            reg_alpha=2.0, reg_lambda=10.0,
            random_state=42, verbose=-1, n_jobs=-1)
        m.fit(X_sparse_tab_all[tr_idx], y_tv_arr[tr_idx])
        probs = m.predict_proba(X_sparse_tab_all[va_idx])
        oof_lgb_sparse[va_idx] = probs
        fold_scores.append(f1_score(y_tv_arr[va_idx], probs.argmax(axis=1), average='micro'))
    oof_f1 = f1_score(y_tv_arr, oof_lgb_sparse.argmax(axis=1), average='micro')
    oof_macro = f1_score(y_tv_arr, oof_lgb_sparse.argmax(axis=1), average='macro')
    print(f"  LightGBM: Micro-F1={oof_f1:.4f}, Macro-F1={oof_macro:.4f}")

# XGBoost on full TF-IDF + Tabular
print("\nXGBoost on full TF-IDF + Tabular:")
oof_xgb_sparse = np.zeros((len(y_tv_arr), NUM_CLASSES))
fold_scores = []
for tr_idx, va_idx in skf.split(X_sparse_tab_all, y_tv_arr):
    m = xgb.XGBClassifier(
        n_estimators=1000, max_depth=8, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.3,
        reg_alpha=2.0, reg_lambda=10.0,
        objective='multi:softprob', num_class=NUM_CLASSES,
        eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1)
    m.fit(X_sparse_tab_all[tr_idx], y_tv_arr[tr_idx])
    probs = m.predict_proba(X_sparse_tab_all[va_idx])
    oof_xgb_sparse[va_idx] = probs
    fold_scores.append(f1_score(y_tv_arr[va_idx], probs.argmax(axis=1), average='micro'))
oof_f1 = f1_score(y_tv_arr, oof_xgb_sparse.argmax(axis=1), average='micro')
oof_macro = f1_score(y_tv_arr, oof_xgb_sparse.argmax(axis=1), average='macro')
print(f"  XGBoost: Micro-F1={oof_f1:.4f}, Macro-F1={oof_macro:.4f}")


# ============================================================
# EXP 3: Ablation - What does Tabular add to Report?
# ============================================================
print("\n" + "="*70)
print("EXPERIMENT 3: Ablation - Report alone vs +Tabular")
print("="*70)

# Report alone (SVD-64)
oof_rep = np.zeros((len(y_tv_arr), NUM_CLASSES))
for tr_idx, va_idx in skf.split(X_svd64_all := np.vstack([X_svd64_train, X_svd64_val]), y_tv_arr):
    m = lgb.LGBMClassifier(n_estimators=500, max_depth=4, learning_rate=0.05,
                            num_leaves=31, random_state=42, verbose=-1, n_jobs=-1)
    m.fit(X_svd64_all[tr_idx], y_tv_arr[tr_idx])
    oof_rep[va_idx] = m.predict_proba(X_svd64_all[va_idx])
rep_f1 = f1_score(y_tv_arr, oof_rep.argmax(axis=1), average='micro')
print(f"Report alone (SVD-64 + LGB):  {rep_f1:.4f}")

# Report + Tabular
oof_comb = np.zeros((len(y_tv_arr), NUM_CLASSES))
for tr_idx, va_idx in skf.split(X_comb_all, y_tv_arr):
    m = lgb.LGBMClassifier(n_estimators=1000, max_depth=6, learning_rate=0.03,
                            num_leaves=63, subsample=0.8, colsample_bytree=0.7,
                            reg_alpha=1.0, reg_lambda=5.0,
                            random_state=42, verbose=-1, n_jobs=-1)
    m.fit(X_comb_all[tr_idx], y_tv_arr[tr_idx])
    oof_comb[va_idx] = m.predict_proba(X_comb_all[va_idx])
comb_f1 = f1_score(y_tv_arr, oof_comb.argmax(axis=1), average='micro')
print(f"Report + Tabular (SVD-64):     {comb_f1:.4f}  (delta: +{comb_f1-rep_f1:.4f})")

# Also test: Report (full TF-IDF) alone
if HAS_LGB:
    oof_rep_full = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_tfidf_all := sp_vstack([X_tfidf_train, X_tfidf_val]).tocsr(), y_tv_arr):
        m = lgb.LGBMClassifier(n_estimators=500, max_depth=5, learning_rate=0.05,
                                num_leaves=31, colsample_bytree=0.3,
                                random_state=42, verbose=-1, n_jobs=-1)
        m.fit(X_tfidf_all[tr_idx], y_tv_arr[tr_idx])
        oof_rep_full[va_idx] = m.predict_proba(X_tfidf_all[va_idx])
    rep_full_f1 = f1_score(y_tv_arr, oof_rep_full.argmax(axis=1), average='micro')
    print(f"Report alone (full TF-IDF+LGB): {rep_full_f1:.4f}")

    oof_comb_full = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_sparse_tab_all, y_tv_arr):
        m = lgb.LGBMClassifier(n_estimators=1000, max_depth=8, learning_rate=0.03,
                                num_leaves=127, subsample=0.8, colsample_bytree=0.3,
                                reg_alpha=2.0, reg_lambda=10.0,
                                random_state=42, verbose=-1, n_jobs=-1)
        m.fit(X_sparse_tab_all[tr_idx], y_tv_arr[tr_idx])
        oof_comb_full[va_idx] = m.predict_proba(X_sparse_tab_all[va_idx])
    comb_full_f1 = f1_score(y_tv_arr, oof_comb_full.argmax(axis=1), average='micro')
    print(f"Report(full TF-IDF) + Tabular: {comb_full_f1:.4f}  (delta: +{comb_full_f1-rep_full_f1:.4f})")


# ============================================================
# EXP 4: CatBoost with deeper tuning
# ============================================================
if HAS_CB:
    print("\n" + "="*70)
    print("EXPERIMENT 4: CatBoost Deep Tuning on Combined Features")
    print("="*70)

    for depth in [6, 8, 10]:
        for lr in [0.03, 0.05]:
            oof_cb = np.zeros((len(y_tv_arr), NUM_CLASSES))
            for tr_idx, va_idx in skf.split(X_comb_all, y_tv_arr):
                m = CatBoostClassifier(
                    iterations=1500, depth=depth, learning_rate=lr,
                    l2_leaf_reg=3.0, random_state=42, verbose=0, thread_count=-1)
                m.fit(X_comb_all[tr_idx], y_tv_arr[tr_idx])
                oof_cb[va_idx] = m.predict_proba(X_comb_all[va_idx])
            f1 = f1_score(y_tv_arr, oof_cb.argmax(axis=1), average='micro')
            macro = f1_score(y_tv_arr, oof_cb.argmax(axis=1), average='macro')
            print(f"  CatBoost d={depth} lr={lr:.2f}: Micro-F1={f1:.4f}, Macro-F1={macro:.4f}")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("GBT + REPORT + TABULAR: KEY RESULTS")
print("="*70)
print("\nNOTE: 96% seems very high. Max achievable is likely 88-90%.")
print("The 96% claim would require:")
print("  1. Much better hyperparameter tuning (e.g. deeper CatBoost)")
print("  2. Better feature engineering (e.g. Report keywords as features)")
print("  3. Potential data leakage (TF-IDF fitted on train+val)")
print("  4. Or Kaggle test set is easier than val set")


GBT ON REPORT + TABULAR COMBINATION
Testing: Can GBT reach 96% with combined features?

--- Building Report TF-IDF features ---
TF-IDF: train=(1983, 1418), val=(283, 1418)
SVD-64: (1983, 64), SVD-128: (1983, 128)

--- Building Tabular features ---
Tabular features: train=(1983, 77)

EXPERIMENT 1: Report SVD-64 + Tabular (~92d)
Combined features: (2266, 141)

Model                |   Micro-F1 |   Macro-F1
---------------------------------------------
XGBoost              |     0.8645 |     0.7452
LightGBM             |     0.8645 |     0.7400

EXPERIMENT 2: Report TF-IDF (full sparse) + Tabular
Sparse combined: (2266, 1495) (TF-IDF 1418 + Tab 77)

LightGBM on full TF-IDF + Tabular (sparse):
  LightGBM: Micro-F1=0.8336, Macro-F1=0.6588

XGBoost on full TF-IDF + Tabular:
  XGBoost: Micro-F1=0.8482, Macro-F1=0.6926

EXPERIMENT 3: Ablation - Report alone vs +Tabular
Report alone (SVD-64 + LGB):  0.8508
Report + Tabular (SVD-64):     0.8645  (delta: +0.0137)
Report alone (full TF-IDF+LGB): 0

In [ ]:
# ============================================================
# GBT + LOCATION ONE-HOT: THE MISSING PIECE
# Key insight from Cell 15: X_eng = Clinical + Location OHE(636d) + Ratios(10d) = 673d
# reaches 77% with sklearn GradientBoosting.
# This cell tests: Can we reach 96% by combining ALL sources with GBT?
# Hypothesis: Location OHE (636d) is the main driver, not Radiomics.
# ============================================================

print("="*70)
print("GBT + LOCATION ONE-HOT + ALL SOURCES")
print("Testing: Can Location OHE + GBT reach 96%?")
print("="*70)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack as sp_hstack, vstack as sp_vstack
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_tv_arr = np.concatenate([y_train, y_val])


# ============================================================
# Build ALL features: Report + Clinical + Location OHE + Radiomics
# ============================================================

# --- Location One-Hot ---
print("\n--- Building Location One-Hot ---")
loc_all = np.concatenate([data['train']['merged']['Location_enc'].values,
                          data['val']['merged']['Location_enc'].values]).reshape(-1, 1)
loc_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_loc_ohe = loc_ohe.fit_transform(loc_all)
print(f"Location OHE: {X_loc_ohe.shape}")

# --- Report TF-IDF + SVD ---
print("\n--- Building Report features ---")
def get_reports(data_dict):
    reports = {}
    for split in ['train', 'val', 'test']:
        df = data_dict[split]['merged'][['case_id']].copy()
        report_df = data_dict[split]['report']
        df = df.merge(report_df[['case_id', 'report']], on='case_id', how='left')
        df['report'] = df['report'].fillna('')
        reports[split] = dict(zip(df['case_id'], df['report']))
    return reports

reports = get_reports(data)
train_texts = [reports['train'][cid] for cid in reports['train'].keys()]
val_texts   = [reports['val'][cid]   for cid in reports['val'].keys()]

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
tfidf.fit(train_texts)
X_tfidf_all = sp_vstack([tfidf.transform(train_texts), tfidf.transform(val_texts)]).tocsr()
print(f"TF-IDF: {X_tfidf_all.shape}")

svd = TruncatedSVD(n_components=128, random_state=42)
svd.fit(X_tfidf_all)
X_svd_all = svd.transform(X_tfidf_all)
print(f"SVD-128: {X_svd_all.shape}")

# --- Clinical + SI (from data loader) ---
print("\n--- Building Clinical features ---")
X_clin_all = np.vstack([data['train']['merged'][['Sex_enc','Age_clean','Age_missing']].values,
                          data['val']['merged'][['Sex_enc','Age_clean','Age_missing']].values]).astype(np.float32)

si_cols = [c for c in data['train']['merged'].columns if 'Signal Intensity' in c]
X_si_all = np.vstack([data['train']['merged'][si_cols].values,
                        data['val']['merged'][si_cols].values]).astype(np.float32)
print(f"Clinical: {X_clin_all.shape}, SI: {X_si_all.shape}")

# --- Radiomics with cross-modal ratios (25d ratios) ---
print("\n--- Building Radiomics with full ratios (25d) ---")
MODALITIES_R = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']
RAD_NAMES = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
           'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']
eps = 1e-8

def build_radiomics_full(data_dict):
    result = {}
    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']
        raw_list = []
        for mod in MODALITIES_R:
            cols = [f'{mod}_{r}' for r in RAD_NAMES]
            vals = merged[cols].fillna(0).values.astype(np.float32)
            raw_list.append(vals)
        raw = np.hstack(raw_list)
        md = {'ax_t1': raw[:,0:5], 'ax_t1c': raw[:,5:10],
              'ax_t2': raw[:,10:15], 'ax_t2f': raw[:,15:20]}
        ratios = []
        for i in range(5):
            t1=md['ax_t1'][:,i]; t1c=md['ax_t1c'][:,i]
            t2=md['ax_t2'][:,i];  t2f=md['ax_t2f'][:,i]
            ratios.extend([
                np.log(np.clip((t1c+eps)/(t1+eps), eps, 100)),
                np.log(np.clip((t2f+eps)/(t2+eps), eps, 100)),
                np.log(np.clip((t1+eps)/(t2+eps), eps, 100)),
                np.clip(t1c - t1, -100, 100),
                np.clip(t2f - t2, -100, 100),
            ])
        ratios = np.column_stack(ratios)
        missing = np.column_stack([
            merged.get(f'{mod}_missing', pd.Series(0,index=merged.index)).values.astype(np.float32)
            for mod in MODALITIES_R])
        result[split] = np.hstack([raw, ratios, missing])
    return result

rad_full = build_radiomics_full(data)
X_rad_all = np.vstack([rad_full['train'], rad_full['val']])
print(f"Radiomics (raw+ratios+missing): {X_rad_all.shape}")


# ============================================================
# EXP 1: Clinical + Location OHE + Ratios (replicate Cell 15)
# ============================================================
print("\n" + "="*70)
print("EXP 1: Clinical + Location OHE + Ratios (replicate Cell 15)")
print("="*70)

X_exp1 = np.hstack([X_clin_all, X_loc_ohe, X_rad_all[:, 20:30]])  # Clinical + OHE + Ratios(10d)
print(f"Features: {X_exp1.shape} (Clin + LocOHE + Ratios10d, shallow trees)")

models_exp1 = []
names_exp1 = ['XGBoost', 'LightGBM']
models_exp1.append(lambda: xgb.XGBClassifier(
    n_estimators=1000, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=1.0,
    reg_alpha=1.0, reg_lambda=5.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1))
if HAS_LGB:
    models_exp1.append(lambda: lgb.LGBMClassifier(
        n_estimators=1000, max_depth=4, learning_rate=0.05,
        num_leaves=15, subsample=0.8, colsample_bytree=1.0,
        reg_alpha=1.0, reg_lambda=5.0,
        random_state=42, verbose=-1, n_jobs=-1))
if HAS_CB:
    names_exp1.append('CatBoost')
    models_exp1.append(lambda: CatBoostClassifier(
        iterations=1500, depth=10, learning_rate=0.05,
        l2_leaf_reg=1.0, random_state=42, verbose=0, thread_count=-1))

print(f"\n{'Model':<20s} | {'Micro-F1':>10s} | {'Macro-F1':>10s}")
print("-" * 45)
for name, factory in zip(names_exp1, models_exp1):
    oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_exp1, y_tv_arr):
        m = factory()
        m.fit(X_exp1[tr_idx], y_tv_arr[tr_idx])
        oof[va_idx] = m.predict_proba(X_exp1[va_idx])
    f1 = f1_score(y_tv_arr, oof.argmax(axis=1), average='micro')
    macro = f1_score(y_tv_arr, oof.argmax(axis=1), average='macro')
    print(f"{name:<20s} | {f1:>10.4f} | {macro:>10.4f}")


# ============================================================
# EXP 2: Report + Clinical + Location OHE + Ratios
# ============================================================
print("\n" + "="*70)
print("EXP 2: Report(SVD-128) + Clinical + Location OHE + Ratios")
print("="*70)

X_exp2 = np.hstack([X_svd_all, X_clin_all, X_loc_ohe, X_rad_all[:, 20:30]])
print(f"Features: {X_exp2.shape}")

models_exp2 = []
names_exp2 = ['XGBoost', 'LightGBM']
models_exp2.append(lambda: xgb.XGBClassifier(
    n_estimators=1000, max_depth=8, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.5,
    reg_alpha=1.0, reg_lambda=5.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1))
if HAS_LGB:
    models_exp2.append(lambda: lgb.LGBMClassifier(
        n_estimators=1000, max_depth=8, learning_rate=0.03,
        num_leaves=127, subsample=0.8, colsample_bytree=0.5,
        reg_alpha=1.0, reg_lambda=5.0,
        random_state=42, verbose=-1, n_jobs=-1))
if HAS_CB:
    names_exp2.append('CatBoost')
    models_exp2.append(lambda: CatBoostClassifier(
        iterations=1500, depth=8, learning_rate=0.03,
        l2_leaf_reg=5.0, random_state=42, verbose=0, thread_count=-1))

print(f"\n{'Model':<20s} | {'Micro-F1':>10s} | {'Macro-F1':>10s}")
print("-" * 45)
for name, factory in zip(names_exp2, models_exp2):
    oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_exp2, y_tv_arr):
        m = factory()
        m.fit(X_exp2[tr_idx], y_tv_arr[tr_idx])
        oof[va_idx] = m.predict_proba(X_exp2[va_idx])
    f1 = f1_score(y_tv_arr, oof.argmax(axis=1), average='micro')
    macro = f1_score(y_tv_arr, oof.argmax(axis=1), average='macro')
    print(f"{name:<20s} | {f1:>10.4f} | {macro:>10.4f}")


# ============================================================
# EXP 3: ALL SOURCES (Report + Clinical + LocOHE + Radiomics raw)
# ============================================================
print("\n" + "="*70)
print("EXP 3: ALL SOURCES (Report SVD-128 + Clin + LocOHE + RadFull)")
print("="*70)

X_exp3 = np.hstack([X_svd_all, X_clin_all, X_loc_ohe, X_rad_all])
print(f"Features: {X_exp3.shape}")

models_exp3 = []
names_exp3 = ['XGBoost', 'LightGBM']
models_exp3.append(lambda: xgb.XGBClassifier(
    n_estimators=1000, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.4,
    reg_alpha=2.0, reg_lambda=10.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1))
if HAS_LGB:
    models_exp3.append(lambda: lgb.LGBMClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.03,
        num_leaves=63, subsample=0.8, colsample_bytree=0.4,
        reg_alpha=2.0, reg_lambda=10.0,
        random_state=42, verbose=-1, n_jobs=-1))
if HAS_CB:
    names_exp3.append('CatBoost')
    models_exp3.append(lambda: CatBoostClassifier(
        iterations=1500, depth=6, learning_rate=0.03,
        l2_leaf_reg=10.0, random_state=42, verbose=0, thread_count=-1))

print(f"\n{'Model':<20s} | {'Micro-F1':>10s} | {'Macro-F1':>10s}")
print("-" * 45)
for name, factory in zip(names_exp3, models_exp3):
    oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_exp3, y_tv_arr):
        m = factory()
        m.fit(X_exp3[tr_idx], y_tv_arr[tr_idx])
        oof[va_idx] = m.predict_proba(X_exp3[va_idx])
    f1 = f1_score(y_tv_arr, oof.argmax(axis=1), average='micro')
    macro = f1_score(y_tv_arr, oof.argmax(axis=1), average='macro')
    print(f"{name:<20s} | {f1:>10.4f} | {macro:>10.4f}")


# ============================================================
# EXP 4: Ablation - What does Location OHE contribute?
# ============================================================
print("\n" + "="*70)
print("EXP 4: Ablation - Location OHE contribution")
print("="*70)

# Without Location OHE
X_no_loc = np.hstack([X_svd_all, X_clin_all, X_rad_all[:, 20:30]])
print(f"Without Loc OHE: {X_no_loc.shape}")
if HAS_LGB:
    oof_noloc = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_no_loc, y_tv_arr):
        m = lgb.LGBMClassifier(n_estimators=1000, max_depth=8, learning_rate=0.03,
                               num_leaves=127, subsample=0.8, colsample_bytree=0.5,
                               reg_alpha=1.0, reg_lambda=5.0, random_state=42, verbose=-1, n_jobs=-1)
        m.fit(X_no_loc[tr_idx], y_tv_arr[tr_idx])
        oof_noloc[va_idx] = m.predict_proba(X_no_loc[va_idx])
    f1_noloc = f1_score(y_tv_arr, oof_noloc.argmax(axis=1), average='micro')
    print(f"  LGB without Loc OHE: Micro-F1={f1_noloc:.4f}")

# With Location OHE
oof_loc = np.zeros((len(y_tv_arr), NUM_CLASSES))
for tr_idx, va_idx in skf.split(X_exp2, y_tv_arr):
    m = lgb.LGBMClassifier(n_estimators=1000, max_depth=8, learning_rate=0.03,
                           num_leaves=127, subsample=0.8, colsample_bytree=0.5,
                           reg_alpha=1.0, reg_lambda=5.0, random_state=42, verbose=-1, n_jobs=-1)
    m.fit(X_exp2[tr_idx], y_tv_arr[tr_idx])
    oof_loc[va_idx] = m.predict_proba(X_exp2[va_idx])
f1_loc = f1_score(y_tv_arr, oof_loc.argmax(axis=1), average='micro')
print(f"  LGB with Loc OHE: Micro-F1={f1_loc:.4f}")
if HAS_LGB:
    print(f"  Location OHE contribution: +{f1_loc-f1_noloc:.4f}")


# ============================================================
# EXP 5: CatBoost deeper tuning (best so far)
# ============================================================
if HAS_CB:
    print("\n" + "="*70)
    print("EXP 5: CatBoost Deep Tuning (ALL sources)")
    print("="*70)

    for depth in [6, 8, 10]:
        for lr in [0.03, 0.05]:
            for l2 in [3.0, 5.0, 10.0]:
                oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
                for tr_idx, va_idx in skf.split(X_exp3, y_tv_arr):
                    m = CatBoostClassifier(
                        iterations=2000, depth=depth, learning_rate=lr,
                        l2_leaf_reg=l2, random_state=42, verbose=0, thread_count=-1)
                    m.fit(X_exp3[tr_idx], y_tv_arr[tr_idx])
                    oof[va_idx] = m.predict_proba(X_exp3[va_idx])
                f1 = f1_score(y_tv_arr, oof.argmax(axis=1), average='micro')
                macro = f1_score(y_tv_arr, oof.argmax(axis=1), average='macro')
                if f1 > 0.80:
                    print(f"  CatBoost d={depth} lr={lr:.2f} l2={l2}: Micro={f1:.4f}, Macro={macro:.4f} **")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("SUMMARY: GBT + Location OHE Results")
print("="*70)
print("\nNote: 96% seems unrealistic for this dataset.")
print("  - Max achievable with proper CV: ~86-88%")
print("  - Cell 15 (sklearn GB, Clinical+LocOHE+Ratios): 77%")
print("  - Location OHE (636d) is the KEY discriminator")
print("  - Adding Report pushes toward 85%+")
print("  - 96% likely requires data leakage or test set differences")
print("\n[Location OHE experiment cell ready]")


GBT + LOCATION ONE-HOT + ALL SOURCES
Testing: Can Location OHE + GBT reach 96%?

--- Building Location One-Hot ---
Location OHE: (2266, 572)

--- Building Report features ---
TF-IDF: (2266, 1418)
SVD-128: (2266, 128)

--- Building Clinical features ---
Clinical: (2266, 3), SI: (2266, 24)

--- Building Radiomics with full ratios (25d) ---
Radiomics (raw+ratios+missing): (2266, 49)

EXP 1: Clinical + Location OHE + Ratios (replicate Cell 15)
Features: (2266, 585) (Clin + LocOHE + Ratios10d, shallow trees)

Model                |   Micro-F1 |   Macro-F1
---------------------------------------------
XGBoost              |     0.5763 |     0.3869
LightGBM             |     0.5349 |     0.3148
CatBoost             |     0.6271 |     0.3939

EXP 2: Report(SVD-128) + Clinical + Location OHE + Ratios
Features: (2266, 713)

Model                |   Micro-F1 |   Macro-F1
---------------------------------------------
XGBoost              |     0.8614 |     0.7278
LightGBM             |     0.8623 

In [ ]:
# ============================================================
# NLP LOCATION FEATURES vs CROSS-MODAL RATIOS COMPARISON
# Compare three approaches:
# 1. NLP-extracted Location keywords (~20d) - from model_training approach
# 2. Cross-modal Radiomics Ratios (25d) - from model_training_new approach
# 3. Combination: NLP Location + Cross-modal Ratios
# ============================================================

print("="*70)
print("NLP LOCATION vs CROSS-MODAL RATIOS COMPARISON")
print("="*70)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_tv_arr = np.concatenate([y_train, y_val])


# ============================================================
# METHOD 1: NLP-extracted Location Features (from model_training)
# ============================================================
print("\n--- Building NLP Location Features ---")

def extract_core_anatomy(text):
    """Extract anatomical keywords from location text."""
    if pd.isna(text) or text == 'unknown' or text == 'nan':
        return []
    text = str(text).lower()
    # Skip if too long (likely a paragraph, not a location)
    if len(text) > 200:
        return []

    extracted = set()

    # Laterality
    if 'left' in text or '左' in text: extracted.add('left')
    if 'right' in text or '右' in text: extracted.add('right')
    if 'bilateral' in text or '双' in text:
        extracted.add('left')
        extracted.add('right')

    # Lobes
    if 'frontal' in text or '额' in text: extracted.add('frontal')
    if 'temporal' in text or '颞' in text: extracted.add('temporal')
    if 'parietal' in text or '顶' in text: extracted.add('parietal')
    if 'occipital' in text or '枕' in text: extracted.add('occipital')
    if 'insular' in text or 'insula' in text or '岛' in text: extracted.add('insular')

    # Deep structures
    if 'cerebellum' in text or 'cerebellar' in text or '小脑' in text: extracted.add('cerebellum')
    if 'ventricle' in text or '脑室' in text: extracted.add('ventricle')
    if 'brainstem' in text or 'pons' in text or 'medulla' in text or '脑干' in text: extracted.add('brainstem')
    if 'thalamus' in text or '丘脑' in text: extracted.add('thalamus')
    if 'basal ganglia' in text or '基底节' in text: extracted.add('basal_ganglia')
    if 'corpus callosum' in text or '胼胝体' in text: extracted.add('corpus_callosum')

    # Specific regions
    if 'sellar' in text or 'sella' in text or '鞍' in text: extracted.add('sellar_region')
    if 'pineal' in text or '松果体' in text: extracted.add('pineal_region')
    if 'falx' in text or '大脑镰' in text: extracted.add('falx_cerebri')
    if 'mening' in text or '脑膜' in text: extracted.add('meninges')
    if 'cranial fossa' in text or '颅窝' in text: extracted.add('cranial_fossa')
    if 'cerebellopontine' in text or 'cp angle' in text or '桥小脑' in text: extracted.add('cp_angle')
    if 'supratentorial' in text: extracted.add('supratentorial')
    if 'infratentorial' in text: extracted.add('infratentorial')
    if 'parasagittal' in text: extracted.add('parasagittal')
    if 'convexity' in text: extracted.add('convexity')

    return list(extracted)

# Build NLP location features
merged_all = pd.concat([data['train']['merged'], data['val']['merged']], ignore_index=True)
loc_texts = merged_all['Tumor_Location_raw'].astype(str).fillna('unknown')
loc_lists = loc_texts.apply(extract_core_anatomy)

mlb = MultiLabelBinarizer()
X_loc_nlp = mlb.fit_transform(loc_lists)
print(f"NLP Location features: {X_loc_nlp.shape}")
print(f"  Keywords: {list(mlb.classes_)}")

# Also build One-Hot for comparison
from sklearn.preprocessing import OneHotEncoder
loc_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_loc_ohe = loc_ohe.fit_transform(merged_all['Location_enc'].values.reshape(-1, 1))
print(f"One-Hot Location: {X_loc_ohe.shape}")


# ============================================================
# METHOD 2: Cross-modal Radiomics Ratios (from model_training_new)
# ============================================================
print("\n--- Building Cross-modal Radiomics Ratios ---")

MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']
RAD_NAMES = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
             'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']
eps = 1e-8

def build_cross_modal_ratios(data_dict):
    """Build 25d cross-modal ratio features (model_training_new approach)."""
    result = {}
    for split in ['train', 'val', 'test']:
        merged = data_dict[split]['merged']

        # Raw radiomics (20d)
        raw_list = []
        for mod in MODALITIES:
            cols = [f'{mod}_{r}' for r in RAD_NAMES]
            vals = merged[cols].fillna(0).values.astype(np.float32)
            raw_list.append(vals)
        raw = np.hstack(raw_list)

        # Per-modality arrays
        md = {
            'ax_t1': raw[:, 0:5], 'ax_t1c': raw[:, 5:10],
            'ax_t2': raw[:, 10:15], 'ax_t2f': raw[:, 15:20]
        }

        # 5 ratio types × 5 features = 25d
        ratios = []
        for i in range(5):
            t1 = md['ax_t1'][:, i]
            t1c = md['ax_t1c'][:, i]
            t2 = md['ax_t2'][:, i]
            t2f = md['ax_t2f'][:, i]

            ratios.append(np.log(np.clip((t1c + eps) / (t1 + eps), eps, 100)))  # T1c/T1
            ratios.append(np.log(np.clip((t2f + eps) / (t2 + eps), eps, 100)))  # T2F/T2
            ratios.append(np.log(np.clip((t1 + eps) / (t2 + eps), eps, 100)))   # T1/T2
            ratios.append(np.clip(t1c - t1, -100, 100))                          # Delta T1c
            ratios.append(np.clip(t2f - t2, -100, 100))                          # Delta T2f

        ratios = np.column_stack(ratios)

        # Missing flags (4d)
        missing = np.column_stack([
            merged.get(f'{mod}_missing', pd.Series(0, index=merged.index)).values.astype(np.float32)
            for mod in MODALITIES
        ])

        result[split] = np.hstack([raw, ratios, missing])  # 20 + 25 + 4 = 49d
    return result

rad_ratios = build_cross_modal_ratios(data)
X_rad_all = np.vstack([rad_ratios['train'], rad_ratios['val']])
print(f"Radiomics (raw+ratios+missing): {X_rad_all.shape}")

# Extract just the 25d ratios for comparison
X_ratios_only = X_rad_all[:, 20:45]
print(f"Cross-modal ratios only: {X_ratios_only.shape}")


# ============================================================
# METHOD 3: Clinical + SI features
# ============================================================
print("\n--- Building Clinical + SI features ---")

X_clin_all = merged_all[['Sex_enc', 'Age_clean', 'Age_missing']].values.astype(np.float32)
si_cols = [c for c in merged_all.columns if 'Signal Intensity' in c]
X_si_all = merged_all[si_cols].values.astype(np.float32)
X_clin_si = np.hstack([X_clin_all, X_si_all])
print(f"Clinical + SI: {X_clin_si.shape}")


# ============================================================
# EXPERIMENT 1: Location NLP vs Location OHE
# ============================================================
print("\n" + "="*70)
print("EXP 1: Location Feature Comparison (NLP vs OHE)")
print("="*70)

configs_loc = [
    ("Clinical + SI + Loc NLP", np.hstack([X_clin_si, X_loc_nlp])),
    ("Clinical + SI + Loc OHE", np.hstack([X_clin_si, X_loc_ohe])),
]

for name, X in configs_loc:
    print(f"\n{name}: {X.shape}")

    models = [lambda: xgb.XGBClassifier(n_estimators=700, max_depth=10, learning_rate=0.2,
                                        subsample=1.0, random_state=42, n_jobs=-1)]
    model_names = ['XGB']

    if HAS_LGB:
        models.append(lambda: lgb.LGBMClassifier(n_estimators=1000, max_depth=10, learning_rate=0.05,
                                                  num_leaves=127, random_state=42, verbose=-1, n_jobs=-1))
        model_names.append('LGB')

    print(f"  {'Model':<10s} | {'Micro-F1':>10s}")
    print("  " + "-" * 25)
    for m_name, factory in zip(model_names, models):
        oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
        for tr_idx, va_idx in skf.split(X, y_tv_arr):
            m = factory()
            m.fit(X[tr_idx], y_tv_arr[tr_idx])
            oof[va_idx] = m.predict_proba(X[va_idx])
        f1 = f1_score(y_tv_arr, oof.argmax(axis=1), average='micro')
        print(f"  {m_name:<10s} | {f1:>10.4f}")


# ============================================================
# EXPERIMENT 2: Radiomics Ratios contribution
# ============================================================
print("\n" + "="*70)
print("EXP 2: Cross-modal Ratios Contribution")
print("="*70)

configs_rad = [
    ("Raw Radiomics (20d)", X_rad_all[:, :20]),
    ("Raw + Ratios (45d)", X_rad_all[:, :45]),
    ("Ratios only (25d)", X_ratios_only),
    ("Full Rad (49d)", X_rad_all),
]

for name, X in configs_rad:
    oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X, y_tv_arr):
        m = xgb.XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)
        m.fit(X[tr_idx], y_tv_arr[tr_idx])
        oof[va_idx] = m.predict_proba(X[va_idx])
    f1 = f1_score(y_tv_arr, oof.argmax(axis=1), average='micro')
    print(f"  {name:<25s}: Micro-F1={f1:.4f}")


# ============================================================
# EXPERIMENT 3: Combined - NLP Location + Cross-modal Ratios
# ============================================================
print("\n" + "="*70)
print("EXP 3: Combined Features (NLP Loc + Cross-modal Ratios)")
print("="*70)

configs_comb = [
    ("Clin+SI + LocNLP + Ratios", np.hstack([X_clin_si, X_loc_nlp, X_ratios_only])),
    ("Clin+SI + LocOHE + Ratios", np.hstack([X_clin_si, X_loc_ohe, X_ratios_only])),
    ("Clin+SI + LocNLP + FullRad", np.hstack([X_clin_si, X_loc_nlp, X_rad_all])),
    ("Clin+SI + LocOHE + FullRad", np.hstack([X_clin_si, X_loc_ohe, X_rad_all])),
]

print(f"\n{'Configuration':<35s} | {'XGB':>8s} | {'LGB':>8s} | {'CB':>8s}")
print("-" * 65)

for name, X in configs_comb:
    scores = []

    # XGB
    oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X, y_tv_arr):
        m = xgb.XGBClassifier(n_estimators=1000, max_depth=8, learning_rate=0.03,
                              subsample=0.8, colsample_bytree=0.7,
                              reg_alpha=1.0, reg_lambda=5.0,
                              random_state=42, n_jobs=-1)
        m.fit(X[tr_idx], y_tv_arr[tr_idx])
        oof[va_idx] = m.predict_proba(X[va_idx])
    scores.append(f1_score(y_tv_arr, oof.argmax(axis=1), average='micro'))

    # LGB
    if HAS_LGB:
        oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
        for tr_idx, va_idx in skf.split(X, y_tv_arr):
            m = lgb.LGBMClassifier(n_estimators=1000, max_depth=8, learning_rate=0.03,
                                   num_leaves=127, subsample=0.8, colsample_bytree=0.7,
                                   reg_alpha=1.0, reg_lambda=5.0,
                                   random_state=42, verbose=-1, n_jobs=-1)
            m.fit(X[tr_idx], y_tv_arr[tr_idx])
            oof[va_idx] = m.predict_proba(X[va_idx])
        scores.append(f1_score(y_tv_arr, oof.argmax(axis=1), average='micro'))
    else:
        scores.append(float('nan'))

    # CatBoost
    if HAS_CB:
        oof = np.zeros((len(y_tv_arr), NUM_CLASSES))
        for tr_idx, va_idx in skf.split(X, y_tv_arr):
            m = CatBoostClassifier(iterations=1000, depth=8, learning_rate=0.03,
                                   l2_leaf_reg=5.0, random_state=42, verbose=0)
            m.fit(X[tr_idx], y_tv_arr[tr_idx])
            oof[va_idx] = m.predict_proba(X[va_idx])
        scores.append(f1_score(y_tv_arr, oof.argmax(axis=1), average='micro'))
    else:
        scores.append(float('nan'))

    print(f"{name:<35s} | {scores[0]:>8.4f} | {scores[1]:>8.4f} | {scores[2]:>8.4f}")


# ============================================================
# EXPERIMENT 4: With Report (the full stack)
# ============================================================
print("\n" + "="*70)
print("EXP 4: Adding Report (TF-IDF SVD-64)")
print("="*70)

# Build Report SVD-64
train_texts = [data['train']['report'].set_index('case_id')['report'].get(cid, '')
               for cid in data['train']['merged']['case_id']]
val_texts = [data['val']['report'].set_index('case_id')['report'].get(cid, '')
             for cid in data['val']['merged']['case_id']]

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
tfidf.fit(train_texts)
X_tfidf_all = tfidf.transform(train_texts + val_texts)

from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=64, random_state=42)
X_rep_svd = svd.fit_transform(X_tfidf_all)
print(f"Report SVD-64: {X_rep_svd.shape}")

# Best config from EXP3 + Report
X_full = np.hstack([X_rep_svd, X_clin_si, X_loc_nlp, X_ratios_only])
print(f"\nFull stack (Rep+Clin+LocNLP+Ratios): {X_full.shape}")

oof_full = np.zeros((len(y_tv_arr), NUM_CLASSES))
for tr_idx, va_idx in skf.split(X_full, y_tv_arr):
    m = xgb.XGBClassifier(n_estimators=1000, max_depth=6, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.5,
                          reg_alpha=2.0, reg_lambda=10.0,
                          random_state=42, n_jobs=-1)
    m.fit(X_full[tr_idx], y_tv_arr[tr_idx])
    oof_full[va_idx] = m.predict_proba(X_full[va_idx])
f1_full = f1_score(y_tv_arr, oof_full.argmax(axis=1), average='micro')
macro_full = f1_score(y_tv_arr, oof_full.argmax(axis=1), average='macro')
print(f"  XGB Full Stack: Micro-F1={f1_full:.4f}, Macro-F1={macro_full:.4f}")

if HAS_LGB:
    oof_full_lgb = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_full, y_tv_arr):
        m = lgb.LGBMClassifier(n_estimators=1000, max_depth=6, learning_rate=0.03,
                               num_leaves=63, subsample=0.8, colsample_bytree=0.5,
                               reg_alpha=2.0, reg_lambda=10.0,
                               random_state=42, verbose=-1, n_jobs=-1)
        m.fit(X_full[tr_idx], y_tv_arr[tr_idx])
        oof_full_lgb[va_idx] = m.predict_proba(X_full[va_idx])
    f1_lgb = f1_score(y_tv_arr, oof_full_lgb.argmax(axis=1), average='micro')
    print(f"  LGB Full Stack: Micro-F1={f1_lgb:.4f}")

if HAS_CB:
    oof_full_cb = np.zeros((len(y_tv_arr), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_full, y_tv_arr):
        m = CatBoostClassifier(iterations=1000, depth=6, learning_rate=0.03,
                               l2_leaf_reg=10.0, random_state=42, verbose=0)
        m.fit(X_full[tr_idx], y_tv_arr[tr_idx])
        oof_full_cb[va_idx] = m.predict_proba(X_full[va_idx])
    f1_cb = f1_score(y_tv_arr, oof_full_cb.argmax(axis=1), average='micro')
    print(f"  CB Full Stack:  Micro-F1={f1_cb:.4f}")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("SUMMARY: NLP Location vs Cross-modal Ratios")
print("="*70)
print("\nKey Findings:")
print("  1. NLP Location (~20d) vs OHE (636d): Check which is better")
print("  2. Cross-modal Ratios (25d): Should add ~2-3% over raw radiomics")
print("  3. Combined (NLP Loc + Ratios): Best tabular-only configuration")
print("  4. With Report: Should push toward 85-88%")
print("\nModel Training Approaches Compared:")
print("  - model_training:       NLP Location + Clinical + SI")
print("  - model_training_new:   Cross-modal Ratios + Clinical + SI")
print("  - This cell:            Combined approach (NLP + Ratios + Clinical)")


NLP LOCATION vs CROSS-MODAL RATIOS COMPARISON

--- Building NLP Location Features ---
NLP Location features: (2266, 22)
  Keywords: ['basal_ganglia', 'brainstem', 'cerebellum', 'convexity', 'corpus_callosum', 'cp_angle', 'cranial_fossa', 'falx_cerebri', 'frontal', 'insular', 'left', 'meninges', 'occipital', 'parasagittal', 'parietal', 'pineal_region', 'right', 'sellar_region', 'supratentorial', 'temporal', 'thalamus', 'ventricle']
One-Hot Location: (2266, 572)

--- Building Cross-modal Radiomics Ratios ---
Radiomics (raw+ratios+missing): (2266, 49)
Cross-modal ratios only: (2266, 25)

--- Building Clinical + SI features ---
Clinical + SI: (2266, 27)

EXP 1: Location Feature Comparison (NLP vs OHE)

Clinical + SI + Loc NLP: (2266, 49)
  Model      |   Micro-F1
  -------------------------
  XGB        |     0.7983
  LGB        |     0.7913

Clinical + SI + Loc OHE: (2266, 599)
  Model      |   Micro-F1
  -------------------------
  XGB        |     0.7436
  LGB        |     0.6915

EXP 2

## 5. Report Experiments

In [ ]:

print("\n" + "="*70)
print("REPORT EXPERIMENTS")
print("="*70)

# Fit TF-IDF on train only (no leakage)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X_tfidf_train = tfidf.fit_transform(rep_data['train']['texts'])
X_tfidf_val   = tfidf.transform(rep_data['val']['texts'])
X_tfidf_tv    = np.vstack([X_tfidf_train.toarray(), X_tfidf_val.toarray()])
print(f"TF-IDF: {X_tfidf_tv.shape}")

# SVD dimension search
best_svd_dim = 64
for svd_dim in [32, 64, 128, 256]:
    svd = TruncatedSVD(n_components=svd_dim, random_state=42)
    X_svd = np.vstack([svd.fit_transform(X_tfidf_train), svd.transform(X_tfidf_val)])
    print(f"\n--- SVD-{svd_dim} ({svd.explained_variance_ratio_.sum()*100:.1f}% variance) ---")
    models = [(n, f, s) for n, f, s in get_models_report() if f() is not None]
    results = cv_evaluate(f'report_svd{svd_dim}', X_svd, y_tv, models)

# Use best SVD dim
svd = TruncatedSVD(n_components=best_svd_dim, random_state=42)
X_svd_train = svd.fit_transform(X_tfidf_train)
X_svd_val   = svd.transform(X_tfidf_val)
X_svd       = np.vstack([X_svd_train, X_svd_val])

print(f"\n--- Report FULL (SVD-{best_svd_dim}) ---")
models = [(n, f, s) for n, f, s in get_models_report() if f() is not None]
results = cv_evaluate('report', X_svd, y_tv, models)
rep_results = results

sorted_models = sorted(results.items(), key=lambda x: -x[1]['micro'])
top3 = [m[0] for m in sorted_models[:3]]
rep_ens_oof = np.mean([results[n]['oof_probs'] for n in top3], axis=0)
preds = rep_ens_oof.argmax(axis=1)
print(f"\n  report TOP-3: {top3}")
print(f"    Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")



REPORT EXPERIMENTS
TF-IDF: (2266, 1418)

--- SVD-32 (57.5% variance) ---
  report_svd32              / SVM-RBF-C10         : Micro=0.8288, Macro=0.6831, Wtd=0.8229
  report_svd32              / SVM-RBF-C100        : Micro=0.8283, Macro=0.7057, Wtd=0.8240
  report_svd32              / LGB-d4-lr05         : Micro=0.8394, Macro=0.7146, Wtd=0.8346
  report_svd32              / XGB-d5-lr03         : Micro=0.8195, Macro=0.6601, Wtd=0.8112

--- SVD-64 (71.1% variance) ---
  report_svd64              / SVM-RBF-C10         : Micro=0.8429, Macro=0.6929, Wtd=0.8373
  report_svd64              / SVM-RBF-C100        : Micro=0.8433, Macro=0.7195, Wtd=0.8393
  report_svd64              / LGB-d4-lr05         : Micro=0.8442, Macro=0.7203, Wtd=0.8395
  report_svd64              / XGB-d5-lr03         : Micro=0.8402, Macro=0.6979, Wtd=0.8342

--- SVD-128 (83.4% variance) ---
  report_svd128             / SVM-RBF-C10         : Micro=0.8411, Macro=0.7035, Wtd=0.8364
  report_svd128             / SVM-RBF-C1

## 5b. Report — Frozen BERT Embeddings + MLP


In [ ]:

print("="*70)
print("REPORT — FROZEN BERT EMBEDDINGS + MLP")
print("="*70)

from transformers import BertTokenizer, BertModel
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load frozen BERT
print("Loading BioClinicalBERT (frozen)...")
bert_model = BertModel.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')
bert_model.eval()
bert_model.to(device)

# Freeze everything
for param in bert_model.parameters():
    param.requires_grad = False

tokenizer = BertTokenizer.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')
MAX_LEN = 128

def get_bert_embeddings(texts, batch_size=32):
    """Extract CLS-pooled BERT embeddings (frozen, no fine-tuning)."""
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts, padding=True, truncation=True,
            max_length=MAX_LEN, return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        with torch.no_grad():
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            # CLS token pooling
            cls_embed = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeds.append(cls_embed)
    return np.vstack(all_embeds)

# Extract embeddings
print("Extracting BERT embeddings for train+val...")
X_bert_train = get_bert_embeddings(rep_data['train']['texts'])
X_bert_val   = get_bert_embeddings(rep_data['val']['texts'])
X_bert       = np.vstack([X_bert_train, X_bert_val])
print(f"BERT embeddings: {X_bert.shape} (768-dim)")

# Also extract for test
print("Extracting BERT embeddings for test...")
X_bert_test = get_bert_embeddings(rep_data['test']['texts'])
print(f"Test BERT embeddings: {X_bert_test.shape}")

# CV with MLP (scaled)
print("\n--- BERT + MLP CV ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
mlp_results = {}

for name, hidden, alpha in [
    ('MLP-256', (256,), 0.001),
    ('MLP-256-128', (256, 128), 0.001),
    ('MLP-512-256', (512, 256), 0.0001),
]:
    micro_s, macro_s = [], []
    oof_probs = np.zeros((len(y_tv), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_bert, y_tv):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_bert[tr_idx])
        X_va = scaler.transform(X_bert[va_idx])
        y_tr, y_va = y_tv[tr_idx], y_tv[va_idx]

        clf = MLPClassifier(
            hidden_layer_sizes=hidden, activation='relu', alpha=alpha,
            max_iter=500, early_stopping=True, random_state=42
        )
        clf.fit(X_tr, y_tr)
        probs = clf.predict_proba(X_va)
        oof_probs[va_idx] = probs
        micro_s.append(f1_score(y_va, probs.argmax(axis=1), average='micro'))
        macro_s.append(f1_score(y_va, probs.argmax(axis=1), average='macro'))

    oof_preds = oof_probs.argmax(axis=1)
    print(f"  {name:15s}: Micro={f1_score(y_tv, oof_preds, average='micro'):.4f}, Macro={f1_score(y_tv, oof_preds, average='macro'):.4f}")
    mlp_results[name] = {
        'micro': f1_score(y_tv, oof_preds, average='micro'),
        'macro': f1_score(y_tv, oof_preds, average='macro'),
        'oof_probs': oof_probs
    }

# Also try XGBoost on BERT embeddings
print("\n--- BERT + XGBoost CV ---")
for name, depth, lr, reg in [
    ('XGB-d3', 3, 0.05, 1.0),
    ('XGB-d5', 5, 0.03, 2.0),
    ('XGB-d9', 9, 0.02, 2.0)
]:
    micro_s, macro_s = [], []
    oof_probs = np.zeros((len(y_tv), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_bert, y_tv):
        clf = xgb.XGBClassifier(
            n_estimators=800, max_depth=depth, learning_rate=lr,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=reg, reg_lambda=reg*2,
            objective='multi:softprob', num_class=NUM_CLASSES,
            eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1
        )
        clf.fit(X_bert[tr_idx], y_tv[tr_idx])
        probs = clf.predict_proba(X_bert[va_idx])
        oof_probs[va_idx] = probs
        micro_s.append(f1_score(y_tv[va_idx], probs.argmax(axis=1), average='micro'))
        macro_s.append(f1_score(y_tv[va_idx], probs.argmax(axis=1), average='macro'))

    oof_preds = oof_probs.argmax(axis=1)
    print(f"  {name:15s}: Micro={f1_score(y_tv, oof_preds, average='micro'):.4f}, Macro={f1_score(y_tv, oof_preds, average='macro'):.4f}")
    mlp_results[name] = {
        'micro': f1_score(y_tv, oof_preds, average='micro'),
        'macro': f1_score(y_tv, oof_preds, average='macro'),
        'oof_probs': oof_probs
    }

# Top-3 ensemble
sorted_models = sorted(mlp_results.items(), key=lambda x: -x[1]['micro'])
top3 = [m[0] for m in sorted_models[:3]]
bert_ens_oof = np.mean([mlp_results[n]['oof_probs'] for n in top3], axis=0)
preds = bert_ens_oof.argmax(axis=1)
print(f"\n  BERT+MLP TOP-3 ENSEMBLE: {top3}")
print(f"    Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")

# Save BERT embeddings for final training
print("\nBERT embeddings extracted. Will be used in final submission.")


REPORT — FROZEN BERT EMBEDDINGS + MLP
Device: cuda
Loading BioClinicalBERT (frozen)...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Extracting BERT embeddings for train+val...
BERT embeddings: (2266, 768) (768-dim)
Extracting BERT embeddings for test...
Test BERT embeddings: (572, 768)

--- BERT + MLP CV ---
  MLP-256        : Micro=0.8147, Macro=0.6790
  MLP-256-128    : Micro=0.8182, Macro=0.6683
  MLP-512-256    : Micro=0.8169, Macro=0.6647

--- BERT + XGBoost CV ---
  XGB-d3         : Micro=0.8257, Macro=0.6991
  XGB-d5         : Micro=0.8252, Macro=0.6906
  XGB-d9         : Micro=0.8217, Macro=0.6887

  BERT+MLP TOP-3 ENSEMBLE: ['XGB-d3', 'XGB-d5', 'XGB-d9']
    Micro=0.8248, Macro=0.6992

BERT embeddings extracted. Will be used in final submission.


## 6. Location Experiments

In [ ]:

print("\n" + "="*70)
print("LOCATION EXPERIMENTS")
print("="*70)

X_loc = np.concatenate([loc_data['train']['features'], loc_data['val']['features']]).reshape(-1, 1)
print(f"Location cardinality: {X_loc.max()+1}")

models = [(n, f, s) for n, f, s in get_models_radiomics() if f() is not None]
results = cv_evaluate('location', X_loc, y_tv, models)
loc_results = results

sorted_models = sorted(results.items(), key=lambda x: -x[1]['micro'])
top3 = [m[0] for m in sorted_models[:3]]
loc_ens_oof = np.mean([results[n]['oof_probs'] for n in top3], axis=0)
preds = loc_ens_oof.argmax(axis=1)
print(f"\n  location TOP-3: {top3}")
print(f"    Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")



LOCATION EXPERIMENTS
Location cardinality: 636
  location                  / XGB-d6-lr05         : Micro=0.6169, Macro=0.5049, Wtd=0.5901
  location                  / LGB-d6-lr05         : Micro=0.6262, Macro=0.5266, Wtd=0.5988

  location TOP-3: ['LGB-d6-lr05', 'XGB-d6-lr05']
    Micro=0.6267, Macro=0.5256


## 6.5. Class Weight Experiment (Imbalanced Data)

In [ ]:

print("\n" + "="*70)
print("CLASS WEIGHT EXPERIMENT (imbalanced data)")
print("="*70)

# Balanced class weights
cw = compute_class_weight('balanced', classes=np.unique(y_tv), y=y_tv)
cw_dict = dict(zip(np.unique(y_tv), cw))
print("Balanced class weights:", {k: f'{v:.2f}' for k, v in cw_dict.items()})

# Test best models with class weights
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for source_name, X_source, source_results in [
    ('radiomics (all4)', X_rad_all, rad_all_results),
    ('image', X_img, img_results),
    ('clinical', X_clin, clin_results),
]:
    # Find top model (handle different key names: 'micro' or 'oof_f1')
    score_key = 'micro' if 'micro' in list(source_results.values())[0] else 'oof_f1'
    best_model_name = max(source_results.items(), key=lambda x: x[1][score_key])[0]
    print(f"\n--- {source_name}: best model = {best_model_name} (Micro={source_results[best_model_name][score_key]:.4f}) ---")

    # With class weights
    micro_w, macro_w = [], []
    oof_w = np.zeros((len(y_tv), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_source, y_tv):
        sw = compute_sample_weight('balanced', y_tv[tr_idx])
        clf = xgb.XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0, objective='multi:softprob', num_class=NUM_CLASSES, eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1)
        clf.fit(X_source[tr_idx], y_tv[tr_idx], sample_weight=sw)
        probs = clf.predict_proba(X_source[va_idx])
        oof_w[va_idx] = probs
        micro_w.append(f1_score(y_tv[va_idx], probs.argmax(axis=1), average='micro'))
        macro_w.append(f1_score(y_tv[va_idx], probs.argmax(axis=1), average='macro'))

    print(f"  WITH class weights: Micro={np.mean(micro_w):.4f}, Macro={np.mean(macro_w):.4f}")

    # Without class weights
    micro_nw, macro_nw = [], []
    oof_nw = np.zeros((len(y_tv), NUM_CLASSES))
    for tr_idx, va_idx in skf.split(X_source, y_tv):
        clf = xgb.XGBClassifier(n_estimators=800, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0, objective='multi:softprob', num_class=NUM_CLASSES, eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1)
        clf.fit(X_source[tr_idx], y_tv[tr_idx])
        probs = clf.predict_proba(X_source[va_idx])
        oof_nw[va_idx] = probs
        micro_nw.append(f1_score(y_tv[va_idx], probs.argmax(axis=1), average='micro'))
        macro_nw.append(f1_score(y_tv[va_idx], probs.argmax(axis=1), average='macro'))

    print(f"  WITHOUT class weights: Micro={np.mean(micro_nw):.4f}, Macro={np.mean(macro_nw):.4f}")



CLASS WEIGHT EXPERIMENT (imbalanced data)
Balanced class weights: {np.int64(0): '1.57', np.int64(1): '0.43', np.int64(2): '0.54', np.int64(3): '17.43', np.int64(4): '7.08'}

--- radiomics (all4): best model = radall_XGB-d6-lr05 (Micro=0.5093) ---
  WITH class weights: Micro=0.4991, Macro=0.3071
  WITHOUT class weights: Micro=0.5252, Macro=0.2641

--- image: best model = XGBoost (Micro=0.5984) ---
  WITH class weights: Micro=0.5843, Macro=0.2985
  WITHOUT class weights: Micro=0.5984, Macro=0.2799

--- clinical: best model = XGB-d6-lr05 (Micro=0.6598) ---
  WITH class weights: Micro=0.5437, Macro=0.3988
  WITHOUT class weights: Micro=0.6593, Macro=0.4569


## 7. Cross-Source Blending

In [ ]:
# ============================================================
# ENHANCED RADIOMICS + IMAGE - SIMPLIFIED
# Only XGBoost and LightGBM (strongest models)
# ============================================================

print("="*70)
print("ENHANCED RADIOMICS + IMAGE (Simplified)")
print("="*70)

import sys
sys.path.insert(0, os.getcwd())

# Build unified features
print("\n--- Building unified features ---")
unified, feat_list = build_unified_features(data, clin_data, loc_data, img_data,
                                             use_enhanced_rad=True, use_img_stats=True)

X_unified = unified['train']['X']
y_unified = unified['train']['y']
X_test = unified['test']['X']

print(f"\nUnified: {X_unified.shape}")
print(f"  Components: {' + '.join(f[0] for f in feat_list)}")

# Only XGBoost and LightGBM
print("\n--- 5-Fold CV (XGBoost + LightGBM) ---")

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

try:
    import lightgbm as lgb
    HAS_LGB = True
except:
    HAS_LGB = False

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof = np.zeros((len(y_unified), NUM_CLASSES))
if HAS_LGB:
    lgb_oof = np.zeros((len(y_unified), NUM_CLASSES))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_unified, y_unified)):
    X_tr, X_va = X_unified[tr_idx], X_unified[va_idx]
    y_tr = y_unified[tr_idx]
    sw = compute_sample_weight('balanced', y_tr)

    # XGBoost
    xgb_clf = xgb.XGBClassifier(
        n_estimators=800, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.5,
        reg_alpha=1.0, reg_lambda=5.0,
        objective='multi:softprob', num_class=NUM_CLASSES,
        eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1
    )
    xgb_clf.fit(X_tr, y_tr, sample_weight=sw)
    xgb_oof[va_idx] = xgb_clf.predict_proba(X_va)

    # LightGBM
    if HAS_LGB:
        lgb_clf = lgb.LGBMClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.05, num_leaves=31,
            subsample=0.8, colsample_bytree=0.5,
            reg_alpha=1.0, reg_lambda=5.0,
            random_state=42, verbose=-1, n_jobs=-1
        )
        lgb_clf.fit(X_tr, y_tr, sample_weight=sw)
        lgb_oof[va_idx] = lgb_clf.predict_proba(X_va)

    xgb_f1 = f1_score(y_unified[va_idx], xgb_oof[va_idx].argmax(axis=1), average='micro')
    print(f"  Fold {fold+1}: XGB={xgb_f1:.4f}", end="")
    if HAS_LGB:
        lgb_f1 = f1_score(y_unified[va_idx], lgb_oof[va_idx].argmax(axis=1), average='micro')
        print(f", LGB={lgb_f1:.4f}")
    else:
        print()

# Final results
xgb_f1 = f1_score(y_unified, xgb_oof.argmax(axis=1), average='micro')
print(f"\nXGBoost OOF Micro-F1: {xgb_f1:.4f}")

if HAS_LGB:
    lgb_f1 = f1_score(y_unified, lgb_oof.argmax(axis=1), average='micro')
    print(f"LightGBM OOF Micro-F1: {lgb_f1:.4f}")

    # Ensemble both
    ensemble_oof = 0.6 * xgb_oof + 0.4 * lgb_oof
    ens_f1 = f1_score(y_unified, ensemble_oof.argmax(axis=1), average='micro')
    print(f"Ensemble (0.6XGB+0.4LGB) OOF Micro-F1: {ens_f1:.4f}")

# Per-class accuracy
print(f"\nPer-class (XGBoost):")
for i, cls in enumerate(label_encoder.classes_):
    mask = (y_unified == i)
    if mask.sum() > 0:
        acc = (xgb_oof[mask].argmax(axis=1) == i).mean()
        print(f"  {str(cls)[:40]:40s}: acc={acc:.4f} (n={mask.sum()})")

# Save for blending
unified_xgb_oof = xgb_oof
if HAS_LGB:
    unified_lgb_oof = lgb_oof
    unified_ens_oof = ensemble_oof

print("\nSaved unified_oof for blending")


ENHANCED RADIOMICS + IMAGE (Simplified)

--- Building unified features ---

Unified: (2266, 1242)
  Components: clinical + location + radiomics + image + img_stats

--- 5-Fold CV (XGBoost + LightGBM) ---
  Fold 1: XGB=0.6872, LGB=0.6718
  Fold 2: XGB=0.7020, LGB=0.6954
  Fold 3: XGB=0.7064, LGB=0.7020
  Fold 4: XGB=0.6733, LGB=0.6512
  Fold 5: XGB=0.7020, LGB=0.7020

XGBoost OOF Micro-F1: 0.6942
LightGBM OOF Micro-F1: 0.6845
Ensemble (0.6XGB+0.4LGB) OOF Micro-F1: 0.6884

Per-class (XGBoost):
  Brain Metastase Tumour                  : acc=0.2188 (n=288)
  Glioma                                  : acc=0.8428 (n=1056)
  Meningioma                              : acc=0.7356 (n=832)
  Pineal tumour and Choroid plexus tumour : acc=0.0000 (n=26)
  Tumors of the sellar region             : acc=0.1250 (n=64)

Saved unified_oof for blending


In [ ]:

print("\n" + "="*70)
print("CROSS-SOURCE BLENDING")
print("="*70)

# Collect all ensemble OOF probs
# Collect all ensemble OOF probs (handle missing variables)
source_oof = {
    'rad_t1':      rad_ens_oof.get('ax_t1') if 'rad_ens_oof' in globals() else None,
    'rad_t1c':     rad_ens_oof.get('ax_t1c') if 'rad_ens_oof' in globals() else None,
    'rad_t2':      rad_ens_oof.get('ax_t2') if 'rad_ens_oof' in globals() else None,
    'rad_t2f':     rad_ens_oof.get('ax_t2f') if 'rad_ens_oof' in globals() else None,
    'rad_all':     rad_ens_oof.get('radall') if 'rad_ens_oof' in globals() else None,
    'img_rad':     img_rad_ens_oof if 'img_rad_ens_oof' in globals() else None,
    'clinical':    clin_ens_oof if 'clin_ens_oof' in globals() else None,
    'image':       img_ens_oof if 'img_ens_oof' in globals() else None,
    'report':      rep_ens_oof if 'rep_ens_oof' in globals() else None,
    'report_bert': bert_ens_oof if 'bert_ens_oof' in globals() else None,
    'location':    loc_ens_oof if 'loc_ens_oof' in globals() else None,
}

# Remove None entries
source_oof = {k: v for k, v in source_oof.items() if v is not None}
print(f"Available sources: {list(source_oof.keys())}")
source_oof = {k: v for k, v in source_oof.items() if v is not None}

print(f"\nSources: {list(source_oof.keys())}")
print("\n--- Individual source performance ---")
for name, probs in source_oof.items():
    preds = probs.argmax(axis=1)
    print(f"  {name:<15}: Micro={f1_score(y_tv, preds, average='micro'):.4f}, Macro={f1_score(y_tv, preds, average='macro'):.4f}")



CROSS-SOURCE BLENDING
Available sources: ['rad_t1', 'rad_t1c', 'rad_t2', 'rad_t2f', 'rad_all', 'img_rad', 'clinical', 'image', 'report', 'location']

Sources: ['rad_t1', 'rad_t1c', 'rad_t2', 'rad_t2f', 'rad_all', 'img_rad', 'clinical', 'image', 'report', 'location']

--- Individual source performance ---
  rad_t1         : Micro=0.4356, Macro=0.2081
  rad_t1c        : Micro=0.4117, Macro=0.1875
  rad_t2         : Micro=0.4214, Macro=0.2033
  rad_t2f        : Micro=0.4643, Macro=0.2211
  rad_all        : Micro=0.5066, Macro=0.2473
  img_rad        : Micro=0.6094, Macro=0.2956
  clinical       : Micro=0.6602, Macro=0.3991
  image          : Micro=0.5936, Macro=0.2739
  report         : Micro=0.8504, Macro=0.7289
  location       : Micro=0.6267, Macro=0.5256


In [ ]:

from itertools import combinations, product

best_f1 = 0
best_config = None

# Equal-weight combos
for combo_size in [2, 3, 4, 5, 6, 7, 8]:
    for combo in combinations(list(source_oof.keys()), combo_size):
        ens = np.mean([source_oof[s] for s in combo], axis=0)
        preds = ens.argmax(axis=1)
        micro = f1_score(y_tv, preds, average='micro')
        macro = f1_score(y_tv, preds, average='macro')
        if micro > best_f1:
            best_f1 = micro
            best_config = ('equal', combo)
            print(f"  NEW BEST (equal {combo_size}): {combo}")
            print(f"    Micro={micro:.4f}, Macro={macro:.4f}")

# Unequal-weight pairs
for i, s1 in enumerate(list(source_oof.keys())):
    for s2 in list(source_oof.keys())[i+1:]:
        for w1 in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
            w2 = 1 - w1
            ens = w1 * source_oof[s1] + w2 * source_oof[s2]
            preds = ens.argmax(axis=1)
            micro = f1_score(y_tv, preds, average='micro')
            if micro > best_f1:
                best_f1 = micro
                best_config = ('pair', (s1, s2), (w1, w2))
                print(f"  NEW BEST (pair): {s1}({w1:.1f}) + {s2}({w2:.1f})")
                print(f"    Micro={micro:.4f}")

# Unequal-weight triplets
for combo in combinations(list(source_oof.keys()), 3):
    for w1 in [0.2, 0.3, 0.4, 0.5]:
        for w2 in [0.2, 0.3, 0.4]:
            w3 = 1 - w1 - w2
            if w3 >= 0.1:
                ens = w1 * source_oof[combo[0]] + w2 * source_oof[combo[1]] + w3 * source_oof[combo[2]]
                preds = ens.argmax(axis=1)
                micro = f1_score(y_tv, preds, average='micro')
                if micro > best_f1:
                    best_f1 = micro
                    best_config = ('triplet', combo, (w1, w2, w3))
                    print(f"  NEW BEST (triplet): {combo} weights={best_config[2]}")
                    print(f"    Micro={micro:.4f}")

# Unequal-weight quadruplets
for combo in combinations(list(source_oof.keys()), 4):
    for w1 in [0.2, 0.3, 0.4]:
        for w2 in [0.2, 0.3]:
            for w3 in [0.1, 0.2]:
                w4 = 1 - w1 - w2 - w3
                if w4 >= 0.05:
                    ens = (w1 * source_oof[combo[0]] + w2 * source_oof[combo[1]] +
                           w3 * source_oof[combo[2]] + w4 * source_oof[combo[3]])
                    preds = ens.argmax(axis=1)
                    micro = f1_score(y_tv, preds, average='micro')
                    if micro > best_f1:
                        best_f1 = micro
                        best_config = ('quad', combo, (w1, w2, w3, w4))
                        print(f"  NEW BEST (quad): {combo}")
                        print(f"    Micro={micro:.4f}")

print(f"\n{'='*70}")
print(f"BEST BLEND: {best_config}")
print(f"BEST OOF Micro-F1: {best_f1:.4f}")


  NEW BEST (equal 2): ('rad_t1', 'rad_t1c')
    Micro=0.4550, Macro=0.1994
  NEW BEST (equal 2): ('rad_t1', 'rad_t2f')
    Micro=0.4669, Macro=0.2087
  NEW BEST (equal 2): ('rad_t1', 'rad_all')
    Micro=0.5057, Macro=0.2345
  NEW BEST (equal 2): ('rad_t1', 'img_rad')
    Micro=0.5812, Macro=0.2632
  NEW BEST (equal 2): ('rad_t1', 'clinical')
    Micro=0.6293, Macro=0.3124
  NEW BEST (equal 2): ('rad_t1', 'report')
    Micro=0.8332, Macro=0.6406
  NEW BEST (equal 2): ('rad_t2f', 'report')
    Micro=0.8345, Macro=0.6562
  NEW BEST (equal 2): ('clinical', 'report')
    Micro=0.8455, Macro=0.6610
  NEW BEST (equal 2): ('image', 'report')
    Micro=0.8495, Macro=0.6700
  NEW BEST (pair): rad_t1(0.1) + report(0.9)
    Micro=0.8526
  NEW BEST (pair): rad_t1c(0.3) + report(0.7)
    Micro=0.8539
  NEW BEST (pair): img_rad(0.2) + report(0.8)
    Micro=0.8566
  NEW BEST (pair): clinical(0.3) + report(0.7)
    Micro=0.8579

BEST BLEND: ('pair', ('clinical', 'report'), (0.3, 0.7))
BEST OOF Micro-F

In [ ]:

# Reconstruct best ensemble
mode = best_config[0]
if mode == 'equal':
    combo = best_config[1]
    best_ens = np.mean([source_oof[s] for s in combo], axis=0)
elif mode == 'pair':
    s1, s2 = best_config[1]; w1, w2 = best_config[2]
    best_ens = w1 * source_oof[s1] + w2 * source_oof[s2]
elif mode == 'triplet':
    combo, ws = best_config[1], best_config[2]
    best_ens = sum(w * source_oof[s] for s, w in zip(combo, ws))
elif mode == 'quad':
    combo, ws = best_config[1], best_config[2]
    best_ens = sum(w * source_oof[s] for s, w in zip(combo, ws))

best_preds = best_ens.argmax(axis=1)

print(f"\n--- Best Blend Per-Class Accuracy ---")
for i, cls in enumerate(label_encoder.classes_):
    mask = (y_tv == i)
    if mask.sum() > 0:
        acc = (best_preds[mask] == i).mean()
        n = mask.sum()
        print(f"  {str(cls)[:45]:45s}: acc={acc:.4f} (n={n})")

print(f"\nOOF Micro-F1: {f1_score(y_tv, best_preds, average='micro'):.4f}")
print(f"OOF Macro-F1: {f1_score(y_tv, best_preds, average='macro'):.4f}")
print(f"OOF Weighted-F1: {f1_score(y_tv, best_preds, average='weighted'):.4f}")



--- Best Blend Per-Class Accuracy ---
  Brain Metastase Tumour                       : acc=0.5868 (n=288)
  Glioma                                       : acc=0.9205 (n=1056)
  Meningioma                                   : acc=0.9062 (n=832)
  Pineal tumour and Choroid plexus tumour      : acc=0.1154 (n=26)
  Tumors of the sellar region                  : acc=0.7188 (n=64)

OOF Micro-F1: 0.8579
OOF Macro-F1: 0.6896
OOF Weighted-F1: 0.8509


## 8. All Results Summary

In [ ]:

print("="*70)
print("ALL RESULTS SUMMARY")
print("="*70)

all_results = {}
for key, res in rad_all_results.items():
    all_results[key] = res
for key, res in clin_results.items():
    all_results[f'clin_{key}'] = res
for key, res in img_results.items():
    all_results[f'img_{key}'] = res
for key, res in rep_results.items():
    all_results[f'rep_{key}'] = res
for key, res in loc_results.items():
    all_results[f'loc_{key}'] = res

# Handle different key names across result dicts
def get_score(res):
    if 'micro' in res:
        return res['micro']
    elif 'oof_f1' in res:
        return res['oof_f1']
    else:
        return 0.0

sorted_all = sorted(all_results.items(), key=lambda x: -get_score(x[1]))

print(f"\n{'Rank':<5} {'Source':<20} {'Model':<20} {'Micro':>8} {'Macro':>8}")
print("-" * 65)
for rank, (name, res) in enumerate(sorted_all[:30]):
    parts = name.split('_', 1)
    source = parts[0]
    model = parts[1] if len(parts) > 1 else ''
    micro_val = get_score(res)
    macro_val = res.get('macro', res.get('oof_f1', 0.0))
    print(f"{rank+1:<5} {source:<20} {model:<20} {micro_val:>8.4f} {macro_val:>8.4f}")

print(f"\n{'='*70}")
print(f"BEST SINGLE MODEL: {sorted_all[0][0]}")
best_micro = get_score(sorted_all[0][1])
best_macro = sorted_all[0][1].get('macro', sorted_all[0][1].get('oof_f1', 0.0))
print(f"  Micro={best_micro:.4f}, Macro={best_macro:.4f}")
print(f"\nBEST BLEND: Micro={best_f1:.4f}")


ALL RESULTS SUMMARY

Rank  Source               Model                   Micro    Macro
-----------------------------------------------------------------
1     rep                  LGB-d4-lr05            0.8451   0.7222
2     rep                  SVM-RBF-C100           0.8433   0.7195
3     rep                  SVM-RBF-C10            0.8429   0.6929
4     rep                  XGB-d5-lr03            0.8402   0.6979
5     clin                 XGB-d6-lr05            0.6598   0.4123
6     clin                 LGB-d6-lr05            0.6531   0.3773
7     loc                  LGB-d6-lr05            0.6262   0.5266
8     loc                  XGB-d6-lr05            0.6169   0.5049
9     img                  XGBoost                0.5984   0.5984
10    img                  LightGBM               0.5838   0.5838
11    radall               XGB-d6-lr05            0.5093   0.2506
12    radall               LGB-d6-lr05            0.5040   0.2507
13    ax                   t2f_XGB-d6-lr05        0.461

## 9. Final Model Training & Submission

In [ ]:
"""
# ============================================================
# FINAL: TREE-CENTRIC BLEND + PER-CLASS THRESHOLD
# Strategy: Use XGB/LGB on all sources + threshold tuning for weak classes
# ============================================================

print("="*70)
print("FINAL: TREE-CENTRIC + THRESHOLD TUNING")
print("="*70)

test_case_ids = data['test']['merged']['case_id'].values.astype(int)

# ---- Build all feature matrices ----
clin_cols = ['Sex_enc', 'Age_clean', 'Age_missing']
si_cols = [c for c in data['test']['merged'].columns if c.startswith('Signal Intensity')]
X_clin_train = np.vstack([clin_data['train']['features'], clin_data['val']['features']])
X_clin_test = data['test']['merged'][clin_cols + si_cols].fillna(0).values.astype(np.float32)

# Location OHE
loc_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_loc_train = loc_ohe.fit_transform(np.concatenate([loc_data['train']['features'], loc_data['val']['features']]).reshape(-1, 1))
X_loc_test = loc_ohe.transform(data['test']['merged']['Location_enc'].values.astype(int).reshape(-1, 1))

# Report TF-IDF
svd_final = TruncatedSVD(n_components=best_svd_dim, random_state=42)
svd_final.fit(X_tfidf_train)
X_svd_test = svd_final.transform(tfidf.transform(rep_data['test']['texts']))

# Unified features (all sources merged)
X_all_train = np.hstack([X_clin_train, X_loc_train, X_svd, X_bert])
X_all_test = np.hstack([X_clin_test, X_loc_test, X_svd_test, X_bert_test])

print(f"\nFeatures: clin={X_clin_train.shape[1]}, loc={X_loc_train.shape[1]}, svd={X_svd.shape[1]}, bert={X_bert.shape[1]}")
print(f"Unified: {X_all_train.shape}")

# ---- Class weights ----
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
cw = compute_sample_weight('balanced', y_tv)
cw_dict = dict(zip(range(NUM_CLASSES), compute_class_weight('balanced', classes=np.unique(y_tv), y=y_tv)))
print(f"\nClass weights: {', '.join(f'{label_encoder.classes_[k]}={v:.2f}' for k, v in cw_dict.items())}")

# ---- Build OOF probs with tree models on unified features ----
print("\n--- Building OOF probs (tree models on unified) ---")

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# XGBoost with class weight
xgb_oof = np.zeros((len(y_tv), NUM_CLASSES))
for tr_idx, va_idx in skf.split(X_all_train, y_tv):
    X_tr, X_va = X_all_train[tr_idx], X_all_train[va_idx]
    y_tr = y_tv[tr_idx]
    sw = compute_sample_weight('balanced', y_tr)
    clf = xgb.XGBClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.6,
        reg_alpha=1.0, reg_lambda=5.0,
        objective='multi:softprob', num_class=NUM_CLASSES,
        eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1
    )
    clf.fit(X_tr, y_tr, sample_weight=sw)
    xgb_oof[va_idx] = clf.predict_proba(X_va)

# LightGBM with class weight
lgb_oof = np.zeros((len(y_tv), NUM_CLASSES))
for tr_idx, va_idx in skf.split(X_all_train, y_tv):
    X_tr, X_va = X_all_train[tr_idx], X_all_train[va_idx]
    y_tr = y_tv[tr_idx]
    sw = compute_sample_weight('balanced', y_tr)
    clf = lgb.LGBMClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.03, num_leaves=63,
        subsample=0.8, colsample_bytree=0.6,
        reg_alpha=1.0, reg_lambda=5.0,
        random_state=42, verbose=-1, n_jobs=-1
    )
    clf.fit(X_tr, y_tr, sample_weight=sw)
    lgb_oof[va_idx] = clf.predict_proba(X_va)

# Report-only tree (no class weight, since report is dominant)
rep_tree_oof = np.zeros((len(y_tv), NUM_CLASSES))
for tr_idx, va_idx in skf.split(X_svd, y_tv):
    X_tr, X_va = X_svd[tr_idx], X_svd[va_idx]
    y_tr = y_tv[tr_idx]
    clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=2.0, reg_lambda=10.0,
        objective='multi:softprob', num_class=NUM_CLASSES,
        eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1
    )
    clf.fit(X_tr, y_tr)
    rep_tree_oof[va_idx] = clf.predict_proba(X_va)

# Combine OOF probs
tree_oof = 0.5 * xgb_oof + 0.3 * lgb_oof + 0.2 * rep_tree_oof

# Results without threshold
base_preds = tree_oof.argmax(axis=1)
base_f1 = f1_score(y_tv, base_preds, average='micro')
print(f"\nTree blend (no threshold): Micro-F1 = {base_f1:.4f}")

print("\nPer-class accuracy (tree blend, no threshold):")
for i, cls in enumerate(label_encoder.classes_):
    mask = (y_tv == i)
    if mask.sum() > 0:
        acc = (base_preds[mask] == i).mean()
        print(f"  {str(cls)[:40]:40s}: acc={acc:.4f} (n={mask.sum()})")

# ---- Per-class threshold tuning ----
print("\n--- Per-class threshold tuning ---")

def apply_thresholds(probs, thresholds):
    """Apply per-class probability thresholds. If no class exceeds threshold, use argmax."""
    adjusted = np.zeros_like(probs)
    for i in range(len(probs)):
        max_prob = probs[i].max()
        best_class = probs[i].argmax()
        # Only override if max_prob exceeds threshold for that class
        if max_prob > thresholds[best_class]:
            adjusted[i, best_class] = max_prob
        else:
            adjusted[i] = probs[i]
    return adjusted.argmax(axis=1)

# Grid search thresholds for weak classes
# Focus on Brain Metastase (class 0) and Pineal (class 3)
best_f1 = base_f1
best_thresholds = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}

# Only tune thresholds for weak classes
for t0 in [0.0, 0.1, 0.15, 0.2, 0.25]:  # Brain Metastase
    for t3 in [0.0, 0.05, 0.1, 0.15, 0.2]:  # Pineal
        thresholds = {0: t0, 1: 0.0, 2: 0.0, 3: t3, 4: 0.0}
        preds = apply_thresholds(tree_oof, thresholds)
        f1 = f1_score(y_tv, preds, average='micro')
        if f1 > best_f1:
            best_f1 = f1
            best_thresholds = thresholds.copy()
            print(f"  NEW BEST: t0={t0:.2f}, t3={t3:.2f} -> Micro-F1={f1:.4f}")

print(f"\nBest thresholds: {best_thresholds}")
print(f"Best OOF Micro-F1: {best_f1:.4f}")

# Final per-class accuracy with tuned thresholds
final_preds = apply_thresholds(tree_oof, best_thresholds)
print("\nPer-class accuracy (with threshold):")
for i, cls in enumerate(label_encoder.classes_):
    mask = (y_tv == i)
    if mask.sum() > 0:
        old_acc = (base_preds[mask] == i).mean()
        new_acc = (final_preds[mask] == i).mean()
        delta = new_acc - old_acc
        print(f"  {str(cls)[:40]:40s}: {old_acc:.4f} -> {new_acc:.4f} ({delta:+.4f})")

# ---- Train final models on all data ----
print("\n--- Training final models on all data ---")

# XGB final
xgb_final = xgb.XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1
)
xgb_final.fit(X_all_train, y_tv, sample_weight=cw)
xgb_test = xgb_final.predict_proba(X_all_test)

# LGB final
lgb_final = lgb.LGBMClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0,
    random_state=42, verbose=-1, n_jobs=-1
)
lgb_final.fit(X_all_train, y_tv, sample_weight=cw)
lgb_test = lgb_final.predict_proba(X_all_test)

# Report tree final
rep_final = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=2.0, reg_lambda=10.0,
    objective='multi:softprob', num_class=NUM_CLASSES,
    eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1
)
rep_final.fit(X_svd, y_tv)
rep_test = rep_final.predict_proba(X_svd_test)

# Combine test probs
tree_test = 0.5 * xgb_test + 0.3 * lgb_test + 0.2 * rep_test

# Apply thresholds
final_preds_test = apply_thresholds(tree_test, best_thresholds)
final_labels = label_encoder.inverse_transform(final_preds_test)

# ---- Save submissions ----
submission = pd.DataFrame({'case_id': test_case_ids, 'Overall_class': final_labels})
submission.to_csv('submission_tree_blend.csv', index=False)
print(f"\nSaved: submission_tree_blend.csv")
print(f"OOF Micro-F1: {best_f1:.4f}")
print(f"\nTest class distribution:")
print(submission['Overall_class'].value_counts())

# Also save raw (no threshold) for comparison
raw_test_preds = label_encoder.inverse_transform(tree_test.argmax(axis=1))
pd.DataFrame({'case_id': test_case_ids, 'Overall_class': raw_test_preds}).to_csv('submission_tree_raw.csv', index=False)
print(f"Saved: submission_tree_raw.csv")

# Clean up GPU
try:
    del bert_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except: pass

print("\n" + "="*70)
print("ALL DONE")
print("="*70)
"""


FINAL: TREE-CENTRIC + THRESHOLD TUNING

Features: clin=27, loc=572, svd=64, bert=768
Unified: (2266, 1431)

Class weights: Brain Metastase Tumour=1.57, Glioma=0.43, Meningioma=0.54, Pineal tumour and Choroid plexus tumour=17.43, Tumors of the sellar region=7.08

--- Building OOF probs (tree models on unified) ---

Tree blend (no threshold): Micro-F1 = 0.8455

Per-class accuracy (tree blend, no threshold):
  Brain Metastase Tumour                  : acc=0.6458 (n=288)
  Glioma                                  : acc=0.8864 (n=1056)
  Meningioma                              : acc=0.8822 (n=832)
  Pineal tumour and Choroid plexus tumour : acc=0.3462 (n=26)
  Tumors of the sellar region             : acc=0.7969 (n=64)

--- Per-class threshold tuning ---

Best thresholds: {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
Best OOF Micro-F1: 0.8455

Per-class accuracy (with threshold):
  Brain Metastase Tumour                  : 0.6458 -> 0.6458 (+0.0000)
  Glioma                                  : 0.8